# Transformacoes Gold - Siplan RPS

Le as tabelas raw de `lake_prep_siplan` (populadas pelo Dataflow Gen2) e salva as tabelas de producao em `lake_gold_siplan`.

```
lake_prep_siplan   (raw_acoes, raw_tags, raw_datas, raw_projetos,
                    raw_datas_sessoes, raw_acessibilidade,
                    raw_solicitacoes, raw_pcap, raw_pcap_detalhe,
                    raw_parcelas, raw_acoes_txts, raw_pcac_det)
    v  Fabric Notebook (este)
lake_gold_siplan   (tabela_base, datas_sessoes, contratos, pcap)
```

todas_as_datas e datas_sesseoes com problemas
e todas_as_tags?


## Config do ambiente

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd

MONTH_ABBR = {
    1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr', 5: 'mai', 6: 'jun',
    7: 'jul', 8: 'ago', 9: 'set', 10: 'out', 11: 'nov', 12: 'dez',
}

WEEKDAY_ABBR = {0: 'seg', 1: 'ter', 2: 'qua', 3: 'qui',
                4: 'sex', 5: 'sab', 6: 'dom'}

# Datas anteriores a 1900 gravadas por versões antigas do Spark/Hive usam
# calendário Julian; "CORRECTED" trata como Gregoriano sem tentar rebasear.
# Valores serão descartados depois pelo errors='coerce' em pd.to_datetime.
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead",  "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead",     "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInWrite",    "CORRECTED")

def read_raw(table_name: str) -> pd.DataFrame:
    return spark.sql(f'SELECT * FROM lake_prep_siplan.dbo.{table_name}').toPandas()


def read_raw_filtered(table_name: str, ids: pd.Series, dedup_col: str = None, src_col: str = 'atividade_id') -> pd.DataFrame:
    """Lê tabela filtrada por atividade_id via join no Spark.

    dedup_col: se informado, mantém apenas uma linha por esse valor (ROW_NUMBER=1).
    Use para tabelas 1:1 cujo Dataflow acumula cópias a cada refresh (_all mode).
    """
    (
        spark.createDataFrame(ids.drop_duplicates().to_frame())
        .createOrReplaceTempView('_filter_ids')
    )

    _sc = f'`{src_col}`' if '.' in src_col else src_col

    if dedup_col:
        sql = (
            f'SELECT t.* FROM ('
            f'  SELECT /*+ BROADCAST(f) */ t2.*,'
            f'    ROW_NUMBER() OVER (PARTITION BY t2.{dedup_col} ORDER BY t2.{dedup_col}) AS _rn'
            f'  FROM lake_prep_siplan.dbo.{table_name} t2'
            f'  INNER JOIN _filter_ids f ON t2.{_sc} = f.atividade_id'
            f') t WHERE t._rn = 1'
        )
        df = spark.sql(sql).toPandas()
        return df.drop(columns=['_rn'], errors='ignore')

    return spark.sql(
        f'SELECT /*+ BROADCAST(f) */ t.* '
        f'FROM lake_prep_siplan.dbo.{table_name} t '
        f'INNER JOIN _filter_ids f ON t.{_sc} = f.atividade_id'
    ).toPandas()

## 1. Carregar tabelas raw


In [ ]:
HISTORICO = False   # True = _all (histórico completo);  False = só ano corrente
_all = "_all" if HISTORICO else ""

In [ ]:
# Ações, Projetos, Tags e Acessibilidade

raw_acoes_df = read_raw(f'raw_acoes{_all}')
raw_acoes_df['atividade_id'] = raw_acoes_df['atividade_id'].astype(str).str.strip()
raw_acoes_df['servico']      = raw_acoes_df['servico'].astype(str).str.strip()
raw_acoes_df['subatividade'] = raw_acoes_df['subatividade'].astype(str).str.strip()
raw_acoes_df = raw_acoes_df.drop_duplicates(subset=['atividade_id'])
# Normaliza projeto_id (pode vir com prefixo 'a.' do alias SQL)
_acoes_proj_col = next((c for c in raw_acoes_df.columns if c == 'projeto_id' or c.endswith('.projeto_id')), None)
if _acoes_proj_col and _acoes_proj_col != 'projeto_id':
    raw_acoes_df = raw_acoes_df.rename(columns={_acoes_proj_col: 'projeto_id'})
if 'projeto_id' in raw_acoes_df.columns:
    raw_acoes_df['projeto_id'] = raw_acoes_df['projeto_id'].astype(str).str.strip()
print(f'raw_acoes_df:        {raw_acoes_df.shape}')

raw_tags_df = read_raw(f'raw_tags{_all}')
raw_tags_df['atividade_id'] = raw_tags_df['atividade_id'].astype(str).str.strip()
print(f'raw_tags_df:         {raw_tags_df.shape}')

raw_acessibilidade_df = read_raw_filtered(f'raw_acessibilidade{_all}', raw_acoes_df['atividade_id'])
raw_acessibilidade_df['atividade_id'] = raw_acessibilidade_df['atividade_id'].astype(str).str.strip()
print(f'raw_acessibilidade:  {raw_acessibilidade_df.shape}')

In [ ]:
# Diagnóstico: verifica duplicatas por atividade_id em cada tabela _all
# (lightweight — só COUNT, não coleta dados para o driver)
for tbl in ['raw_projetos', 'raw_acessibilidade', 'raw_datas', 'raw_datas_sessoes', 'raw_solicitacoes', 'raw_acoes_txts']:
    spark.sql(f"""
        SELECT
            '{tbl}_all'           AS tabela,
            COUNT(*)              AS total_linhas,
            COUNT(DISTINCT atividade_id) AS atividades_unicas,
            ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT atividade_id), 1) AS media_copias
        FROM lake_prep_siplan.dbo.{tbl}_all
    """).show(truncate=False)

In [ ]:
# Projetos — grain: uma linha por projeto_id
_proj_ids = raw_acoes_df['projeto_id'].dropna().drop_duplicates()
spark.createDataFrame(_proj_ids.to_frame()).createOrReplaceTempView('_filter_proj_ids')

# Detecta nome da coluna no Delta (pode ter prefixo 'p.' do alias SQL)
_rp_cols   = spark.sql(f'SELECT * FROM lake_prep_siplan.dbo.raw_projetos{_all} LIMIT 0').columns
_rp_id_col = next((c for c in _rp_cols if c == 'projeto_id' or c.endswith('.projeto_id')), 'projeto_id')
_rp_id_ref = f'`{_rp_id_col}`' if '.' in _rp_id_col else _rp_id_col

raw_projetos_df = spark.sql(
    f'SELECT /*+ BROADCAST(f) */ t.* '
    f'FROM lake_prep_siplan.dbo.raw_projetos{_all} t '
    f'INNER JOIN _filter_proj_ids f ON t.{_rp_id_ref} = f.projeto_id'
).toPandas()
# Strip de qualquer prefixo de alias SQL (p., pu., etc.)
raw_projetos_df.columns = [c.split('.')[-1] for c in raw_projetos_df.columns]
raw_projetos_df['projeto_id'] = raw_projetos_df['projeto_id'].astype(str).str.strip()
print(f'raw_projetos_df:     {raw_projetos_df.shape}')

In [ ]:
# Datas — 1:1 por atividade; dedup_col descarta cópias do Dataflow
raw_datas_df = read_raw_filtered(f'raw_datas{_all}', raw_acoes_df['atividade_id'], dedup_col='atividade_id')
raw_datas_df['atividade_id'] = raw_datas_df['atividade_id'].astype(str).str.strip()
print(f'raw_datas_df:        {raw_datas_df.shape}')

# datas_sessoes — várias linhas por atividade (uma por sessão); sem dedup
raw_datas_sessoes_raw_df = read_raw_filtered(f'raw_datas_sessoes{_all}', raw_acoes_df['atividade_id'])
print(f'raw_datas_sessoes:   {raw_datas_sessoes_raw_df.shape}')

In [ ]:
# Solicitações
raw_solicitacoes_df = read_raw_filtered(f'raw_solicitacoes{_all}', raw_acoes_df['atividade_id'])
raw_solicitacoes_df['atividade_id']   = raw_solicitacoes_df['atividade_id'].astype(str).str.strip()
raw_solicitacoes_df['solicitacao_id'] = pd.to_numeric(raw_solicitacoes_df['solicitacao_id'], errors='coerce').astype('Int64')
raw_solicitacoes_df['custo']          = pd.to_numeric(raw_solicitacoes_df['custo'], errors='coerce').fillna(0)
raw_solicitacoes_df = raw_solicitacoes_df[
    raw_solicitacoes_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()
print(f'raw_solicitacoes:    {raw_solicitacoes_df.shape}')

In [ ]:
# PCAP
pcap_props_df = read_raw(f'raw_pcap{_all}')
pcap_props_df['pcap_num'] = pd.to_numeric(pcap_props_df['pcap_num'], errors='coerce').astype('Int64')
print(f'raw_pcap:            {pcap_props_df.shape}')

# Parcelas
raw_parcelas_df = read_raw('raw_parcelas')
raw_parcelas_df['solicitacao_id'] = pd.to_numeric(raw_parcelas_df['solicitacao_id'], errors='coerce').astype('Int64')
print(f'raw_parcelas_df:     {raw_parcelas_df.shape}')

# PCAC Detalhe
raw_pcac_det_df = read_raw('raw_pcap_det')
print(f'raw_pcap_det_df:     {raw_pcac_det_df.shape}')


In [ ]:
# Acoes Textos - tem versao _all; filtrado por atividade_id
# coluna de join na fonte tem prefixo 'acao.' (Dataflow)
raw_acoes_txts_df = read_raw_filtered(f'raw_acoes_txts{_all}', raw_acoes_df['atividade_id'], src_col='acao.atividade_id')
raw_acoes_txts_df['acao.atividade_id'] = raw_acoes_txts_df['acao.atividade_id'].astype(str).str.strip()
print(f'raw_acoes_txts_df:   {raw_acoes_txts_df.shape}')

## 2. Tags e RPS Parcial


In [ ]:
# todas_as_tags (calculado em Python — não vem da staging)
todas_tags = (
    raw_tags_df.dropna(subset=['tag_nome'])
    .groupby('atividade_id')['tag_nome']
    .apply(lambda x: ' | '.join(sorted(x.unique())))
    .reset_index(name='todas_as_tags')
)
raw_tags_df = raw_tags_df.drop(columns=['todas_as_tags'], errors='ignore').merge(todas_tags, on='atividade_id', how='left')
print(f'raw_tags_df (com todas_as_tags): {raw_tags_df.shape}')

# rps_parcial_df
sts_df = (
    raw_tags_df[raw_tags_df['tag_nome'] == 'Avaliação STS'][['atividade_id']]
    .assign(tag='Avaliação STS')
    .drop_duplicates()
)
rps_parcial_df = (
    raw_acoes_df[['atividade_id', 'servico', 'subatividade']]
    .merge(sts_df[['atividade_id', 'tag']], on='atividade_id', how='left')
    .assign(tag=lambda d: d['tag'].fillna(''))
    .drop_duplicates(subset=['atividade_id'])
)
print(f'rps_parcial_df: {rps_parcial_df.shape}')

## 3. Precificação e Alerta sobre passagens, hospedagens e PCAP


In [ ]:
# 1d. Parser de precificacao_desc → gratuito, maior_valor, menor_valor
#
# Regras:
#   - Extrai todos os valores "R$ XX.YY" ou "R$ XX,YY" via regex
#   - Detecta "grátis" / "gratuito" / "aberto" no texto (case-insensitive)
#   - gratuito = 'Sim' se não há valor pago (>0); 'Não' caso contrário
#   - maior_valor = max dos valores pagos encontrados (None se tudo grátis ou sem info)
#   - menor_valor = 0 se "grátis/aberto" coexiste com valores pagos; min dos pagos caso contrário

import re

# Ponto ou vírgula como separador decimal: "R$ 10.00" ou "R$ 10,00"
_RE_VALOR  = re.compile(r'R$\s*(\d+[.,]\d{2})', re.IGNORECASE)
# "grátis", "gratis", "gratuito" ou "aberto" → acesso livre
_RE_GRATIS = re.compile(r'gr[áa]tis|gratuito|aberto', re.IGNORECASE)

# -- alertas de complemento (passagem v5) -----------------------------------
import unicodedata as _ud

_SEP_PASS     = r'(?:\s+[xX]\s+|\s*[/\->]\s*)'
_RE_ROTA_COD  = re.compile(
    r'\b[A-Za-z]{2,4}\b' + _SEP_PASS + r'(?:\d+\s*)?\b[A-Za-z]{2,4}\b',
    re.IGNORECASE
)
_RE_ROTA_CIDAD = re.compile(
    r'[A-Za-z]{3,}' + _SEP_PASS + r'[A-Za-z]{2,}',
    re.IGNORECASE
)
_RE_CONCAT_PASS  = re.compile(r'\b[A-Z]{2,4}(?:[xX][A-Z]{2,4})+\b')
_RE_NUM_UNIT_PASS = re.compile(
    r'\b\d+\s*(?:pax|px|pessoa[s]?|aéreas?|passagem|passagens|bilhetes?|trechos?|voos?)\b',
    re.IGNORECASE
)
_RE_KW_PASS = re.compile(
    r'\bpax\b|\bpx\b|\bpassagem\b|\bpassagens\b|\baéreas?\b',
    re.IGNORECASE
)
# Hospedagem: '4pax x 8 diarias' ou '4 x 8 diarias'
_RE_HOSPED_DIARIAS = re.compile(
    r'\b\d+\s*(?:pax|pessoa[s]?|pes\.?)?\s*[xX×]\s*\d+\s*(?:diári[ao]s?|diarias?|noites?)\b',
    re.IGNORECASE
)
# PCAP -- sem grupo de captura (para str.contains)
_RE_PCAP_CHECK = re.compile(r'PCAP[^0-9]*(?:[0-9]{13})', re.IGNORECASE)


def _valid_passagem(text: str) -> bool:
    t = str(text).strip()
    if not t:
        return False
    tn = ''.join(c for c in _ud.normalize('NFD', t) if _ud.category(c) != 'Mn')
    return (
        bool(_RE_ROTA_COD.search(tn))     or
        bool(_RE_ROTA_CIDAD.search(tn))   or
        bool(_RE_CONCAT_PASS.search(t))   or
        bool(_RE_NUM_UNIT_PASS.search(t)) or
        bool(_RE_KW_PASS.search(t))
    )


_RE_HOSP_KW = re.compile(
    r'(?<![a-zA-Z])(?:'
    r'diarias?|diarios?'
    r'|noites?'
    r'|pax|px'
    r'|hospedage(?:m|ns)'
    r'|hospedes?'
    r'|singles?'
    r'|duplos?|dbls?|dpls?'
    r'|triplos?|tpls?'
    r'|doubles?'
    r'|suites?'
    r'|sgls?'
    r'|twn'
    r'|quartos?'
    r'|aptos?'
    r'|hotel'
    r'|pessoas?'
    r')',
    re.IGNORECASE
)
_RE_PLAN_VALID = re.compile(
    r'\bplanilha\s+(?:inserida|anexa|corrigida|atualizada)',
    re.IGNORECASE
)


def _valid_hospedagem(text: str) -> bool:
    t = str(text).strip()
    if not t:
        return False
    tn = ''.join(c for c in _ud.normalize('NFD', t) if _ud.category(c) != 'Mn')
    return bool(_RE_HOSP_KW.search(tn)) or bool(_RE_PLAN_VALID.search(tn))


def _str_to_float(s: str) -> float:
    # Normaliza separador decimal: "10,00" → "10.00", "10.00" → "10.00"
    return float(s.replace(',', '.'))


def parse_precificacao(desc) -> tuple:
    """Retorna (gratuito, maior_valor, menor_valor) a partir de precificacao_desc."""
    if pd.isna(desc) or str(desc).strip() == '':
        return ('Não', None, None)

    s = str(desc)
    tem_gratis = bool(_RE_GRATIS.search(s))

    raw_vals = _RE_VALOR.findall(s)
    valores = []
    for v in raw_vals:
        try:
            valores.append(_str_to_float(v))
        except ValueError:
            pass

    pagos = [v for v in valores if v > 0]

    if not pagos and not tem_gratis:
        return ('Não', None, None)

    if not pagos:
        # Apenas grátis / aberto
        return ('Sim', 0.0, 0.0)

    # Há valores pagos
    maior = max(pagos)
    # "grátis/aberto" coexiste com pago → opção gratuita existe → menor = 0
    menor = 0.0 if tem_gratis else min(pagos)
    return ('Não', maior, menor)


# ── Aplica e exibe diagnóstico ────────────────────────────────────────────────
_res = raw_acoes_df['a.precificacao_desc'].apply(parse_precificacao)
precif_df = pd.DataFrame(_res.tolist(), columns=['gratuito', 'maior_valor', 'menor_valor'],
                         index=raw_acoes_df.index)
precif_df.insert(0, 'atividade_id', raw_acoes_df['atividade_id'])

print("gratuito:")
print(precif_df['gratuito'].value_counts())
print()
print("maior_valor — estatísticas:")
print(precif_df['maior_valor'].describe())
print()
print("menor_valor — estatísticas:")
print(precif_df['menor_valor'].describe())
print()

# Amostra de casos mistos (grátis/aberto + pago na mesma atividade)
_mistos = precif_df[(precif_df['gratuito'] == 'Não') & (precif_df['menor_valor'] == 0)]
print(f"Casos mistos (pago + grátis/aberto): {len(_mistos)}")
if len(_mistos) > 0:
    _check = raw_acoes_df.loc[_mistos.index, ['atividade_id', 'a.precificacao_desc']].head(5)
    for _, row in _check.iterrows():
        print(f"  {row['atividade_id']}: {str(row['a.precificacao_desc'])[:90]}")

## 4. Datas


In [ ]:
def transform_datas(df, rps_df):
    df = df.copy()

    # Remove prefixo de nome de coluna que o driver ODBC pode incluir
    df.columns = [col.split('.')[-1] for col in df.columns]

    # Normaliza atividade_id para string (driver pode retornar float64)
    df['atividade_id'] = df['atividade_id'].astype(str).str.strip()

    # --- data/hora da primeira sessão ---
    df['primeiradata'] = pd.to_datetime(df['primeiradata'], errors='coerce')
    df['PrimeiraData']     = df['primeiradata'].dt.date
    df['PrimeiraHora']     = df['primeiradata'].dt.time
    df['PrimeiraDataHora'] = df['primeiradata'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df['mes'] = df['primeiradata'].dt.month.map(MONTH_ABBR)

    # --- tempo médio da sessão (truncado a 1 decimal) ---
    df['tempo_da_sessao'] = (
        np.floor(pd.to_numeric(df['tempo_sessao'], errors='coerce').fillna(0) * 10) / 10
    )

    # --- flags de prazo (binário → 'sim'/'0') ---
    # 30dias: nº de datas distintas com sessão > 30  (≠ diascorridos > 30)
    # 90dias: diascorridos > 90
    # 60horas: total de horas > 60
    # Todos já calculados no SQL; apenas mapeamos para 'sim'/'0'
    bool_map = {1: 'sim', '1': 'sim', 0: '0', '0': '0'}
    for col in ['30dias', '90dias', '60horas']:
        df[col] = pd.to_numeric(df[col], errors='coerce').map(bool_map).fillna('0')

    df['ExtrapolaDataHora'] = np.where(
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'), 'sim', '0'
    )

    # --- autonomia temporal bruta ---
    # DIREG: ultrapassa 90 dias corridos OU 60 horas totais
    # STS  : mais de 30 datas distintas com sessão
    # UO   : demais casos
    conditions = [
        (df['90dias'] == 'sim') | (df['60horas'] == 'sim'),
        df['30dias'] == 'sim',
    ]
    df['autonomiaTemporal'] = np.select(conditions, ['DIREG', 'STS'], default='UO')

    # --- ajuste por serviço (via RPS Parcial) ---
    # A autonomia temporal só faz sentido para serviços com acúmulo de carga/dias.
    # Para os demais, UO é sempre o nível correto independente das datas.
    df = df.merge(rps_df[['atividade_id', 'servico', 'subatividade']], on='atividade_id', how='left')
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    usa_temporal = (
        df['servico'].isin({'Curso', 'Ioga'}) |
        df['servico'].str.startswith('Desenvolvimento') |
        df['subatividade'].str.startswith('Ações')
    )
    df['autonomiaTemporal'] = np.where(usa_temporal, df['autonomiaTemporal'], 'UO')

    # servico/subatividade já estarão na tabela base; remove daqui para evitar duplicatas
    df = df.drop(columns=['servico', 'subatividade'])
    return df

datas_df = transform_datas(raw_datas_df, rps_parcial_df)
print(f'datas_df: {datas_df.shape}')


## 5. Datas / Sessões


In [ ]:
# Nota: diaSemama preserva o typo original do Power Query (era "semana").
# Renomear quebraria relatórios existentes que referenciem essa coluna.
WEEKDAY_ABBR = {0: "seg", 1: "ter", 2: "qua", 3: "qui",
                4: "sex", 5: "sab", 6: "dom"}

def transform_datas_sessoes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [col.split(".")[-1] for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    g = df['local_grupo']

    # Tipos base
    df["atividade_id"]  = df["atividade_id"].astype(str).str.strip()
    df["sessao_id"]     = df["sessao_id"].astype(str).str.strip()
    df["datainicio"]    = pd.to_datetime(df["datainicio"],    errors="coerce")
    df["datafinal"]     = pd.to_datetime(df["datafinal"],     errors="coerce")
    df["primeira_data"] = pd.to_datetime(df["primeira_data"], errors="coerce")

    # Remove sessões com datas fora do intervalo datetime64[ns] (ex: anos > 2262)
    n_before = len(df)
    df = df[df["datainicio"].notna()].copy()
    n_dropped = n_before - len(df)
    if n_dropped:
        print(f"[AVISO] {n_dropped} sessão(ões) descartada(s): datainicio inválida ou fora dos limites")

    # Exclui grupo "37" (tipologia sem validade)
    df = df[df["local_grupo"] != "37"].copy()

    # Duração e diferença em horas inteiras
    df["Duracao"]   = df["datafinal"] - df["datainicio"]
    df["diferenca"] = (df["Duracao"].dt.total_seconds() / 3600).astype(int)

    # Colunas de tempo derivadas de datainicio
    dt = df["datainicio"].dt
    df["Data"]         = dt.normalize()                        # datetime na meia-noite
    df["HoraCheia"]    = df["datainicio"].dt.floor("h").dt.time  # hora cheia (sem minutos)
    df["horaCerta"]    = dt.time                               # hora exata
    df["horaCertaTxt"] = dt.strftime("%H:%M")

    # PeriodoDia baseado na hora cheia
    hora = dt.hour
    df["PeriodoDia"] = np.select(
        [hora < 5, hora < 12, hora < 18],
        ["Madrugada", "de manhã", "à tarde"],
        default="à noite"
    )

    # TipoLocal — vetorizado (máscara de maior prioridade aplicada por último)
    g = df["local_grupo"].fillna("")
    l = df["local_nome"].fillna("")
    df["TipoLocal"] = "na UO"
    df.loc[g == "Fora da Unidade",              "TipoLocal"] = "externa"
    df.loc[l.str.contains("Online",       na=False), "TipoLocal"] = "online"
    df.loc[l.str.contains("Sesc Digital", na=False), "TipoLocal"] = "online"
    df.loc[g.str.contains("Online",       na=False), "TipoLocal"] = "online"

    # Renomeia e formata texto
    df = df.rename(columns={"local_grupo": "TipologiaLocal", "local_nome": "localNome"})
    df["localNome"] = df["localNome"].str.title()

    # uo = primeiros 2 dígitos do atividade_id
    df["uo"] = df["atividade_id"].str[:2].astype(int)

    # Partes de data (de datainicio)
    df["ano"]           = dt.year
    df["Mes"]           = dt.month
    df["mesTxt"]        = dt.month.map(MONTH_ABBR)
    df["dia"]           = dt.day
    df["diaSemama"]     = dt.dayofweek             # seg=0, dom=6
    df["diaSemanaTxt"]  = dt.dayofweek.map(WEEKDAY_ABBR)
    df["SemanaDoAno"] = dt.isocalendar().week.astype(int)

    # Partes de data (de primeira_data — mês da 1ª sessão da atividade)
    p = df["primeira_data"].dt
    df["mes1o"]    = p.month
    df["mes1oTxt"] = p.month.map(MONTH_ABBR)

    # Remove colunas não consumidas downstream
    df = df.drop(columns=["local_id", "grupo_id", "correcao_local",
                          "uo_local", "geac", "datafinal"], errors="ignore")

    col_order = [
        "atividade_id", "sessao_id", "uo",
        "datainicio", "Data", "diferenca", "Duracao",
        "HoraCheia", "horaCerta", "horaCertaTxt", "PeriodoDia",
        "localNome", "TipologiaLocal", "TipoLocal", "local_grupo",
        "ano", "Mes", "mesTxt", "dia", "diaSemama", "diaSemanaTxt", "SemanaDoAno",
        "primeira_data", "primeira_hora", "mes1o", "mes1oTxt",
    ]
    return df[[c for c in col_order if c in df.columns]]


In [ ]:
raw_datas_sessoes_df = transform_datas_sessoes(raw_datas_sessoes_raw_df)
print(f'raw_datas_sessoes_df: {raw_datas_sessoes_df.shape}')
print(raw_datas_sessoes_df['TipoLocal'].value_counts())


In [ ]:
print(raw_datas_sessoes_raw_df.columns[raw_datas_sessoes_raw_df.columns.duplicated()].tolist())

In [ ]:
# Todas as datas por atividade — formato "seg, dd/mm/yy HHhMM"
# Deduplica por (atividade_id, datainicio) antes de agregar
_d = (
    raw_datas_sessoes_df[['atividade_id', 'datainicio', 'diaSemanaTxt']]
    .drop_duplicates()
    .sort_values(['atividade_id', 'datainicio'])
    .assign(_linha=lambda d:
        d['diaSemanaTxt'] + ', ' + d['datainicio'].dt.strftime('%d/%m/%y %Hh%M')
    )
)

todas_as_datas_df = (
    _d.groupby('atividade_id')['_linha']
    .apply('\n'.join)
    .reset_index(name='todas_as_datas')
)

print(f'todas_as_datas_df: {todas_as_datas_df.shape}')
print()
print('Exemplo:')
print(todas_as_datas_df['todas_as_datas'].iloc[0])

## 6. Projetos


In [ ]:
raw_projetos_df = raw_projetos_df.drop_duplicates(subset=['projeto_id'])
for col in ['projeto_uo_nome']:
    if col in raw_projetos_df.columns:
        raw_projetos_df[col] = raw_projetos_df[col].fillna('').astype(str).str.strip()
print(f'raw_projetos_df: {raw_projetos_df.shape}')


## 7. Acessibilidade


In [ ]:
if 'uo' not in raw_acessibilidade_df.columns:
    raw_acessibilidade_df['uo'] = raw_acessibilidade_df['atividade_id'].str[:2].astype(int)
raw_acessibilidade_df = raw_acessibilidade_df[
    raw_acessibilidade_df['atividade_id'].isin(raw_acoes_df['atividade_id'])
].copy()
print(f'raw_acessibilidade_df: {raw_acessibilidade_df.shape}')


## 8. Solicitações


In [ ]:
# ── tem_passagem: derivado de raw_solicitacoes_df (elimina sql_passagens) ─
# Equivalente a WHERE item_grupo LIKE '%Passagem%'; filtra custo > 0
# (passagens com custo = 0 não devem gerar autonomia STS)
_passagens_ids = raw_solicitacoes_df[
    raw_solicitacoes_df['item_grupo'].str.contains('Passagem', case=False, na=False)
]['atividade_id']

rps_parcial_df['tem_passagem'] = np.where(
    rps_parcial_df['atividade_id'].isin(_passagens_ids), 'Sim', 'Não'
)
print(f'tem_passagem=Sim: {(rps_parcial_df["tem_passagem"] == "Sim").sum()}')

## 9. Contratos


In [ ]:
SERVICOS_LIMIAR_20K  = {'Debate', 'Seminário', 'Visita Mediada'}
SUBATIV_LIMIAR_20K   = {'Ações formativas', 'Ações mediadas', 'Passeios', 'Viagens'}
SERVICOS_POR_HORA    = {'Curso', 'Oficina', 'Vivência', 'Seminário', 'Mediação', 'Visita Mediada', 'Intervenção urbana'}
SUBATIV_POR_HORA     = {'Multipráticas recreativas', 'Passeios', 'Viagens', 'Colônias recreativas'}


def build_contracts(
    raw_solicitacoes_df: pd.DataFrame,
    raw_acoes_df: pd.DataFrame,
    datas_df: pd.DataFrame,
    rps_parcial_df: pd.DataFrame,
) -> pd.DataFrame:
    """Constrói contracts_df a partir de raw_solicitacoes_df + raw_acoes_df.

    Substitui sql_contratos (5 subqueries Hive) por groupby Python sobre
    raw_solicitacoes_df, que já está disponível na seção 8.
    Saída: uma linha por solicitação; métricas agregadas repetidas por atividade_id.
    """
    solic = raw_solicitacoes_df.copy()

    # --- classificação de tipo por item_grupo / nome_item ---
    grupo_lower = solic['item_grupo'].fillna('').str.lower()
    item_lower  = solic['nome_item'].fillna('').str.lower()

    is_admin      = solic['area'].str.strip() == 'Administrativo'
    is_contrato   = grupo_lower.str.contains('contrato') | item_lower.str.contains('contrato')
    is_filme      = grupo_lower.str.contains('filme')
    is_passagem   = grupo_lower.str.contains('passagem') | item_lower.str.contains('passagem')
    is_hospedagem = grupo_lower.str.contains('hospedagem') | item_lower.str.contains('hospedagem')

    # Filtro principal: admin + (contrato | passagem | hospedagem | filme)
    # Espelha o WHERE do FROM principal da query Hive original
    mask_main = is_admin & (is_contrato | is_filme | is_passagem | is_hospedagem)
    df = solic[mask_main].copy()

    # Renomeia para manter compatibilidade com o código consumidor (build_autonomias, base)
    df = df.rename(columns={'item_grupo': 'grupo', 'nome_item': 'item', 'descricao': 'a.complemento'})
    # Remove sufixo "[...]" que a view do Hive às vezes insere no grupo
    df['grupo'] = df['grupo'].astype(str).str.split('[').str[0].str.strip()

    # --- agregados por atividade_id (substituem os 4 subqueries do Hive) ---

    # custo_contratos_total / n_contratos: apenas contrato+filme, área Administrativo
    # Semântica original: base das métricas per-capita (por_sessao, por_hora, per_capita)
    mask_cf = is_admin & (is_contrato | is_filme)
    agg_cf = (
        solic[mask_cf]
        .groupby('atividade_id', as_index=False)
        .agg(custo_contratos_total=('custo', 'sum'), n_contratos=('solicitacao_id', 'count'))
    )

    # custo_total / n_solic: todas as solicitações (raw_solicitacoes_df já filtrou custo > 0)
    agg_total = (
        solic
        .groupby('atividade_id', as_index=False)
        .agg(custo_total=('custo', 'sum'), n_solic=('solicitacao_id', 'count'))
    )

    df = df.merge(agg_cf,    on='atividade_id', how='left')
    df = df.merge(agg_total, on='atividade_id', how='left')
    df[['custo_contratos_total', 'custo_total']] = (
        df[['custo_contratos_total', 'custo_total']].fillna(0)
    )
    df['n_contratos'] = df['n_contratos'].fillna(0).astype(int)
    df['n_solic']     = df['n_solic'].fillna(0).astype(int)

    # --- campos de público (substituem o INNER JOIN com siplan_acao da query Hive) ---
    acoes_pub = raw_acoes_df[['atividade_id', 'a.lugares', 'a.estimativa_publico', 'servico']].copy()

    # publico_sessao: lugares preferido, fallback estimativa_publico, mínimo 1
    # Espelha: NVL(lugares, NVL(estimativa_publico, 1)) com CASE quando <= 0 → 1
    pub_raw = acoes_pub['a.lugares'].where(acoes_pub['a.lugares'].notna(), acoes_pub['a.estimativa_publico'])
    acoes_pub['publico_sessao'] = pd.to_numeric(pub_raw, errors='coerce').fillna(1).clip(lower=1).astype(int)
    acoes_pub['capacidade']     = pd.to_numeric(acoes_pub['a.lugares'],            errors='coerce')
    acoes_pub['estimativa']     = pd.to_numeric(acoes_pub['a.estimativa_publico'], errors='coerce')

    # tipo_per_capita=1 → público é total (não se multiplica por sessões)
    # Espelha: CASE WHEN LOWER(desc_realizacao) IN ('curso', 'seminário') THEN 1 ELSE 0
    acoes_pub['tipo_per_capita'] = acoes_pub['servico'].isin(['Curso', 'Seminário']).astype(int)

    df = df.merge(
        acoes_pub[['atividade_id', 'publico_sessao', 'capacidade', 'estimativa', 'tipo_per_capita']],
        on='atividade_id', how='left',
    )
    df['publico_sessao']  = pd.to_numeric(df['publico_sessao'],  errors='coerce').fillna(1).clip(lower=1).astype(int)
    df['tipo_per_capita'] = pd.to_numeric(df['tipo_per_capita'], errors='coerce').fillna(0)

    # --- sessoes e horas reais (de datas_df, gerado na seção 2) ---
    df = df.merge(datas_df[['atividade_id', 'qt_sessoes', 'qt_horas']], on='atividade_id', how='left')
    df = df.rename(columns={'qt_sessoes': 'sessoes', 'qt_horas': 'horas'})
    df['sessoes'] = pd.to_numeric(df['sessoes'], errors='coerce').fillna(0)
    df['horas']   = pd.to_numeric(df['horas'],   errors='coerce').fillna(0)

    # --- flags de custo INDIVIDUAL (custo desta solicitação, não do total da atividade) ---
    custo_ind = df['custo']
    bool_map  = {1: 'sim', 0: '0'}
    df['acima15mil']  = (custo_ind > 15000).astype(int).map(bool_map)
    df['acima20mil']  = (custo_ind > 20000).astype(int).map(bool_map)
    df['acima100mil'] = (custo_ind > 100000).astype(int).map(bool_map)

    # --- público total estimado da atividade ---
    # tipo_per_capita=0 (Apresentação, Oficina etc.): público repete por sessão → total = sessoes × publico
    # tipo_per_capita=1 (Curso, Seminário): público já é total
    # Acima de 4000: considera total direto (cap para evitar inflação)
    pub = df['publico_sessao'].astype(float)
    tpc = df['tipo_per_capita']
    df['publico'] = np.where(
        pub > 4000, pub,
        np.where(tpc == 0, df['sessoes'] * pub, pub),
    ).astype(int)

    # --- métricas de custo por unidade (baseadas em custo_contratos_total da atividade) ---
    custo_total_ativ = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0)
    df['por_sessao'] = np.where(df['sessoes'] > 0, custo_total_ativ / df['sessoes'], np.nan)
    df['por_hora']   = np.where(df['horas']   > 0, custo_total_ativ / df['horas'],   np.nan)
    df['per_capita'] = np.where(
        tpc == 0,
        np.where(df['publico'] > 0, custo_total_ativ / df['publico'], np.nan),
        np.where(pub > 0, custo_total_ativ / pub, np.nan),
    )

    # --- servico e subatividade para classificar autonomiaCusto ---
    df = df.merge(
        rps_parcial_df[['atividade_id', 'servico', 'subatividade']],
        on='atividade_id', how='left',
    )
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # --- autonomiaCusto baseada no custo INDIVIDUAL desta solicitação ---
    # build_autonomias resolve a hierarquia final tomando drop_duplicates; aqui classificamos por linha
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'

    df['autonomiaCusto'] = np.select(
        [acima100,
         acima20 &  limiar_20k,
         acima20 & ~limiar_20k,
         acima15 & ~limiar_20k],
        ['DIREG', 'STS', 'STS-20', 'STS-15'],
        default='UO',
    )

    # por_hora válido apenas para serviços faturados por horas de execução
    usa_por_hora = (
        df['servico'].isin(SERVICOS_POR_HORA) |
        df['subatividade'].isin(SUBATIV_POR_HORA)
    )
    df['por_hora_valido'] = np.where(usa_por_hora, df['por_hora'], np.nan)

    return df.drop_duplicates(subset=['solicitacao_id'])

In [ ]:
def build_solicitacoes(
    raw_solicitacoes_df: pd.DataFrame,
    raw_acoes_df: pd.DataFrame,
    datas_df: pd.DataFrame,
    rps_parcial_df: pd.DataFrame,
    contracts_df: pd.DataFrame,
) -> pd.DataFrame:
    """Constrói solicitacoes_df: todas as solicitações com custo >= R$100
    mais todas as que já estão em contracts_df (independente do valor).

    Diferenças em relação a contracts_df:
    - Filtro de inclusão mais amplo (valor >= 100 OR já em contracts)
    - sem servico/subatividade no output (estão em tabela_base)
    - custo_solic_filtradas: soma do subconjunto filtrado por atividade_id
    - por_sessao/hora/per_capita → base custo_solic_filtradas
    - por_sessaoC/horaC/per_capitaC → base custo_contratos_total
    """
    solic = raw_solicitacoes_df.copy()

    # ── classificação de tipo ──────────────────────────────────────────────
    grupo_lower = solic['item_grupo'].fillna('').str.lower()
    item_lower  = solic['nome_item'].fillna('').str.lower()

    is_admin      = solic['area'].str.strip() == 'Administrativo'
    is_contrato   = grupo_lower.str.contains('contrato') | item_lower.str.contains('contrato')
    is_filme      = grupo_lower.str.contains('filme')
    is_passagem   = grupo_lower.str.contains('passagem') | item_lower.str.contains('passagem')
    is_hospedagem = grupo_lower.str.contains('hospedagem') | item_lower.str.contains('hospedagem')

    # ── filtro principal: custo >= 100 OU já está em contracts_df ─────────
    contracts_ids     = set(contracts_df['solicitacao_id'])
    mask_100          = pd.to_numeric(solic['custo'], errors='coerce').fillna(0) >= 100
    mask_in_contracts = solic['solicitacao_id'].isin(contracts_ids)
    df = solic[mask_100 | mask_in_contracts].copy()

    df = df.rename(columns={'item_grupo': 'grupo', 'nome_item': 'item', 'descricao': 'a.complemento'})
    df['grupo'] = df['grupo'].astype(str).str.split('[').str[0].str.strip()

    # ── agregados por atividade_id ─────────────────────────────────────────

    # custo_contratos_total / n_contratos: contrato+filme admin (semântica original)
    mask_cf = is_admin & (is_contrato | is_filme)
    agg_cf = (
        solic[mask_cf]
        .groupby('atividade_id', as_index=False)
        .agg(custo_contratos_total=('custo', 'sum'), n_contratos=('solicitacao_id', 'count'))
    )

    # custo_total / n_solic: todas as solicitações
    agg_total = (
        solic
        .groupby('atividade_id', as_index=False)
        .agg(custo_total=('custo', 'sum'), n_solic=('solicitacao_id', 'count'))
    )

    # custo_solic_filtradas: soma das que passaram no filtro (por atividade)
    agg_filtradas = (
        df.groupby('atividade_id', as_index=False)
        .agg(custo_solic_filtradas=('custo', 'sum'))
    )

    df = df.merge(agg_cf,        on='atividade_id', how='left')
    df = df.merge(agg_total,     on='atividade_id', how='left')
    df = df.merge(agg_filtradas, on='atividade_id', how='left')
    df[['custo_contratos_total', 'custo_total', 'custo_solic_filtradas']] = (
        df[['custo_contratos_total', 'custo_total', 'custo_solic_filtradas']].fillna(0)
    )
    df['n_contratos'] = df['n_contratos'].fillna(0).astype(int)
    df['n_solic']     = df['n_solic'].fillna(0).astype(int)

    # ── público (recalculado aqui, não copiado de tabela_base) ────────────
    acoes_pub = raw_acoes_df[['atividade_id', 'a.lugares', 'a.estimativa_publico', 'servico']].copy()
    pub_raw = acoes_pub['a.lugares'].where(
        acoes_pub['a.lugares'].notna(), acoes_pub['a.estimativa_publico']
    )
    acoes_pub['publico_sessao']  = pd.to_numeric(pub_raw, errors='coerce').fillna(1).clip(lower=1).astype(int)
    acoes_pub['capacidade']      = pd.to_numeric(acoes_pub['a.lugares'],            errors='coerce')
    acoes_pub['estimativa']      = pd.to_numeric(acoes_pub['a.estimativa_publico'], errors='coerce')
    acoes_pub['tipo_per_capita'] = acoes_pub['servico'].isin(['Curso', 'Seminário']).astype(int)

    df = df.merge(
        acoes_pub[['atividade_id', 'publico_sessao', 'capacidade', 'estimativa', 'tipo_per_capita']],
        on='atividade_id', how='left',
    )
    df['publico_sessao']  = pd.to_numeric(df['publico_sessao'],  errors='coerce').fillna(1).clip(lower=1).astype(int)
    df['tipo_per_capita'] = pd.to_numeric(df['tipo_per_capita'], errors='coerce').fillna(0)

    # ── sessoes/horas: uso INTERNO para per_* (não vão pro output) ────────
    df = df.merge(datas_df[['atividade_id', 'qt_sessoes', 'qt_horas']], on='atividade_id', how='left')
    _sessoes = pd.to_numeric(df['qt_sessoes'], errors='coerce').fillna(0)
    _horas   = pd.to_numeric(df['qt_horas'],   errors='coerce').fillna(0)
    df = df.drop(columns=['qt_sessoes', 'qt_horas'])

    # ── flags de custo individual ──────────────────────────────────────────
    bool_map = {1: 'sim', 0: '0'}
    df['acima15mil']  = (df['custo'] > 15000 ).astype(int).map(bool_map)
    df['acima20mil']  = (df['custo'] > 20000 ).astype(int).map(bool_map)
    df['acima100mil'] = (df['custo'] > 100000).astype(int).map(bool_map)

    # ── público total da atividade ─────────────────────────────────────────
    pub = df['publico_sessao'].astype(float)
    tpc = df['tipo_per_capita']
    df['publico'] = np.where(
        pub > 4000, pub,
        np.where(tpc == 0, _sessoes * pub, pub),
    ).astype(int)

    # ── servico/subatividade: uso INTERNO para autonomiaCusto e por_hora ──
    df = df.merge(
        rps_parcial_df[['atividade_id', 'servico', 'subatividade']],
        on='atividade_id', how='left',
    )
    df['servico']      = df['servico'].fillna('')
    df['subatividade'] = df['subatividade'].fillna('')

    # ── autonomiaCusto (por solicitação individual) ────────────────────────
    limiar_20k = (
        df['servico'].isin(SERVICOS_LIMIAR_20K) |
        df['subatividade'].isin(SUBATIV_LIMIAR_20K)
    )
    acima15  = df['acima15mil']  == 'sim'
    acima20  = df['acima20mil']  == 'sim'
    acima100 = df['acima100mil'] == 'sim'
    df['autonomiaCusto'] = np.select(
        [acima100,
         acima20 &  limiar_20k,
         acima20 & ~limiar_20k,
         acima15 & ~limiar_20k],
        ['DIREG', 'STS', 'STS-20', 'STS-15'],
        default='UO',
    )

    # ── métricas de custo ─────────────────────────────────────────────────
    c_filt      = pd.to_numeric(df['custo_solic_filtradas'],  errors='coerce').fillna(0)
    c_contratos = pd.to_numeric(df['custo_contratos_total'],  errors='coerce').fillna(0)

    # base: custo_solic_filtradas
    df['por_sessao'] = np.where(_sessoes > 0, c_filt / _sessoes, np.nan)
    df['por_hora']   = np.where(_horas   > 0, c_filt / _horas,   np.nan)
    df['per_capita'] = np.where(
        tpc == 0,
        np.where(df['publico'] > 0, c_filt / df['publico'], np.nan),
        np.where(pub > 0,           c_filt / pub,           np.nan),
    )

    # base: custo_contratos_total
    df['por_sessaoC'] = np.where(_sessoes > 0, c_contratos / _sessoes, np.nan)
    df['por_horaC']   = np.where(_horas   > 0, c_contratos / _horas,   np.nan)
    df['per_capitaC'] = np.where(
        tpc == 0,
        np.where(df['publico'] > 0, c_contratos / df['publico'], np.nan),
        np.where(pub > 0,           c_contratos / pub,           np.nan),
    )

    # por_hora_valido: apenas para serviços faturados por hora de execução
    usa_por_hora = (
        df['servico'].isin(SERVICOS_POR_HORA) |
        df['subatividade'].isin(SUBATIV_POR_HORA)
    )
    df['por_hora_valido']  = np.where(usa_por_hora, df['por_hora'],  np.nan)
    df['por_hora_validoC'] = np.where(usa_por_hora, df['por_horaC'], np.nan)

    # ── remove campos internos (estão em tabela_base) ─────────────────────
    df = df.drop(columns=['servico', 'subatividade'])


    # -- alerta: complemento mal preenchido ------------------------------------
    _grp  = df['grupo'].fillna('').str.lower()
    _itm  = df['item'].fillna('').str.lower()
    _comp = df['a.complemento'].fillna('')

    _is_pass = _grp.str.contains('passagem')   | _itm.str.contains('passagem')
    _is_hosp = _grp.str.contains('hospedagem') | _itm.str.contains('hospedagem')
    _is_cont = _grp.str.contains('contrato')   | _itm.str.contains('contrato')

    _pass_ok = _comp.map(_valid_passagem)
    _hosp_ok = _comp.map(_valid_hospedagem)
    _pcap_ok = _comp.str.contains(_RE_PCAP_CHECK.pattern,     flags=re.IGNORECASE, na=False)

    df['alerta'] = (
        (_is_pass & ~_pass_ok) |
        (_is_hosp & ~_hosp_ok) |
        (_is_cont & ~_pcap_ok)
    ).astype(int)

    return df.drop_duplicates(subset=['solicitacao_id'])


In [ ]:
# contracts_df: tabela de solicitações administrativas com métricas de custo.
# Executado aqui porque build_contracts() depende de:
#   - raw_solicitacoes_df  (disponível após a seção 8)
#   - rps_parcial_df com tem_passagem já adicionado (rps-passagens-exec acima)
contracts_df = build_contracts(
    raw_solicitacoes_df = raw_solicitacoes_df,
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    rps_parcial_df      = rps_parcial_df,
)
print(f'contracts_df:     {contracts_df.shape}')
print(f'atividades unicas: {contracts_df["atividade_id"].nunique()}')
print()
print('autonomiaCusto:')
print(contracts_df.drop_duplicates("atividade_id")["autonomiaCusto"].value_counts())

In [ ]:
# solicitacoes_df: substituto ampliado de contracts_df.
# Inclui todas as solicitações com custo >= R$100 + as que já estavam em contracts_df.
solicitacoes_df = build_solicitacoes(
    raw_solicitacoes_df = raw_solicitacoes_df,
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    rps_parcial_df      = rps_parcial_df,
    contracts_df        = contracts_df,
)
print(f'solicitacoes_df:   {solicitacoes_df.shape}')
print(f'atividades unicas: {solicitacoes_df["atividade_id"].nunique()}')
print()
print('autonomiaCusto:')
print(solicitacoes_df.drop_duplicates("atividade_id")["autonomiaCusto"].value_counts())


## 10. Autonomias


In [ ]:
# Hierarquia: DIREG (3) > STS (2) > UO (1)
# Cada fonte de autonomia e convertida para nivel numerico;
# o nivel maximo determina a autonomia final.


NIVEL = {'DIREG': 3, 'STS': 2, 'STS-20': 2, 'STS-15': 2, 'UO': 1}


def autonomia_nivel(serie: pd.Series) -> pd.Series:
    """Converte valores de autonomia para nivel numerico (desconhecido 1)."""
    return serie.map(NIVEL).fillna(1).astype(int)


NIVEL_INV = {3: 'DIREG', 2: 'STS', 1: 'UO'}


def build_autonomias(rps_df, datas_df, contracts_df, raw_pcap_df) -> pd.DataFrame:

    # Base: uma linha por atividade com servico cadastrado
    df = rps_df[['atividade_id', 'servico', 'subatividade', 'tag', 'tem_passagem']].copy()

    # --- autonomia temporal (ja ajustada por servico em Datas) ---
    df = df.merge(
        datas_df[['atividade_id', 'autonomiaTemporal', '60horas', '90dias', '30dias', 'diascorridos']],
        on='atividade_id', how='left'
    )
    df['autonomiaTemporal'] = df['autonomiaTemporal'].fillna('UO')
    df['60horas']           = df['60horas'].fillna('0')
    df['90dias']            = df['90dias'].fillna('0')
    df['30dias']            = df['30dias'].fillna('0')
    df['diascorridos']      = pd.to_numeric(df['diascorridos'], errors='coerce').fillna(0)

    # --- autonomia de custo (uma linha por atividade, prevalece o maior nivel) ---
    custo_por_ativ = (
        contracts_df[['atividade_id', 'autonomiaCusto']]
        .assign(_nivel=lambda d: d['autonomiaCusto'].map(NIVEL).fillna(1))
        .sort_values('_nivel', ascending=False)
        .drop_duplicates(subset=['atividade_id'])
        .drop(columns=['_nivel'])
    )
    df = df.merge(custo_por_ativ, on='atividade_id', how='left')
    df['autonomiaCusto'] = df['autonomiaCusto'].fillna('UO')

    # --- autonomia por PCAP ---
    # Para cada atividade, aplica os mesmos limiares de autonomiaCusto ao pcap_total de cada PCAP vinculada.
    # Se ao menos uma PCAP atingir um limiar, o nivel mais alto entre todas as PCAPs e considerado.
    _pcap_aut = pd.DataFrame(columns=['atividade_id', 'autonomiaPCAP'])
    if raw_pcap_df is not None and len(raw_pcap_df) > 0 and 'pcap_total' in raw_pcap_df.columns:
        _pcap_vals = raw_pcap_df[['atividade_id', 'pcap_total']].copy()
        _pcap_vals['pcap_total'] = pd.to_numeric(_pcap_vals['pcap_total'], errors='coerce').fillna(0)
        _pcap_ext = (
            df[['atividade_id', 'servico', 'subatividade']]
            .merge(_pcap_vals, on='atividade_id', how='inner')
        )
        _limiar_20k_p = (
            _pcap_ext['servico'].isin(SERVICOS_LIMIAR_20K) |
            _pcap_ext['subatividade'].isin(SUBATIV_LIMIAR_20K)
        )
        _pcap_ext['autonomiaPCAP'] = np.select(
            [_pcap_ext['pcap_total'] > 100_000,
             (_pcap_ext['pcap_total'] > 20_000) &  _limiar_20k_p,
             (_pcap_ext['pcap_total'] > 20_000) & ~_limiar_20k_p,
             (_pcap_ext['pcap_total'] > 15_000) & ~_limiar_20k_p],
            ['DIREG', 'STS', 'STS-20', 'STS-15'],
            default='UO',
        )
        _pcap_aut = (
            _pcap_ext[['atividade_id', 'autonomiaPCAP']]
            .assign(_nivel=lambda d: d['autonomiaPCAP'].map(NIVEL).fillna(1))
            .sort_values('_nivel', ascending=False)
            .drop_duplicates(subset=['atividade_id'])
            .drop(columns=['_nivel'])
        )
    df = df.merge(_pcap_aut, on='atividade_id', how='left')
    df['autonomiaPCAP'] = df['autonomiaPCAP'].fillna('UO')

    # --- regra de custo para 60h (baseada em contratos; PCAP nao altera o rebaixamento) ---
    custo_acima20 = df['autonomiaCusto'].isin(['STS', 'STS-20', 'DIREG'])

    mask_rebaixa = ~custo_acima20 & (df['autonomiaTemporal'] == 'DIREG') & (df['90dias'] != 'sim')
    _sts = (
        (df.loc[mask_rebaixa, '30dias'] == 'sim') |
        (df.loc[mask_rebaixa, 'diascorridos'] > 30)
    )
    df.loc[mask_rebaixa, 'autonomiaTemporal'] = np.where(_sts, 'STS', 'UO')

    # --- niveis por fonte ---
    nivel_temporal  = autonomia_nivel(df['autonomiaTemporal'])
    nivel_custo     = autonomia_nivel(df['autonomiaCusto'])
    nivel_pcap      = autonomia_nivel(df['autonomiaPCAP'])
    nivel_tag       = np.where(df['tag'] == 'Avaliacao STS', 2, 1)
    nivel_passagem  = np.where(df['tem_passagem'] == 'Sim',  2, 1)

    # --- regra combinada: 60h + custo >= 20k -> DIREG ---
    nivel_combinado = np.where(
        (df['60horas'] == 'sim') & custo_acima20,
        3, 1
    )

    # --- autonomia final: prevalece o maior nivel (DIREG > STS > UO) ---
    nivel_final = pd.concat(
        [nivel_temporal, nivel_custo, nivel_pcap,
         pd.Series(nivel_tag,       index=df.index),
         pd.Series(nivel_passagem,  index=df.index),
         pd.Series(nivel_combinado, index=df.index)],
        axis=1
    ).max(axis=1)

    df['autonomia'] = nivel_final.map(NIVEL_INV)

    # --- diagnostico: atividades cuja autonomia foi elevada pelo valor da PCAP ---
    _nivel_sem_pcap = pd.concat(
        [nivel_temporal, nivel_custo,
         pd.Series(nivel_tag,       index=df.index),
         pd.Series(nivel_passagem,  index=df.index),
         pd.Series(nivel_combinado, index=df.index)],
        axis=1
    ).max(axis=1)
    _elev = nivel_pcap > _nivel_sem_pcap
    _n_direg_pcap = int((_elev & (nivel_final == 3)).sum())
    _n_sts_pcap   = int((_elev & (nivel_final == 2)).sum())
    print(f'Via PCAP -> DIREG: {_n_direg_pcap} atividade(s)')
    print(f'Via PCAP -> STS:   {_n_sts_pcap} atividade(s)')

    return df.drop(columns=['60horas', '90dias', '30dias', 'diascorridos'])

## 11. Descrição de Custos


In [ ]:
# Itens a descartar antes de construir item_desc
ITENS_DESCARTAR = frozenset([
    'Camarim', 'Camarim Tipo 1', 'Camarim Tipo 2',
    'Água', 'Verificar',
])

# Mapeamento exato: item_de_custo → categoria normalizada
MAPA_ITEM_CUSTO = {
    # Contratos
    'Contrato PJ [Custo cachê/pró-labore]':            'Contrato PJ',
    'Contrato PJ':                                      'Contrato PJ',
    'Contrato PF [Custo cachê/pró-labore]':            'Contrato PF',
    'Contrato PF':                                      'Contrato PF',
    'Contrato Cooperativa [Custo cachê/pró-labore]':   'Contrato Cooperativa',
    # Viagem
    'Hospedagem [Custo hospedagem]':                   'Hospedagem',
    'Passagem Aérea [Custo passagem]':                 'Passagem Aérea',
    'Turismo [Outros custos de terceiros]':            'Turismo',
    'Turismo':                                          'Turismo',
    'Transporte [Outros custos de terceiros]':         'Transporte',
    'Transporte':                                       'Transporte',
    # Serviços artísticos/técnicos
    'Exibição de Filmes':                              'Exibição de Filmes',
    'Ação esportiva e recreativa':                     'Ação esportiva e recreativa',
    'Contratações diversas [Outros custos de terceiros]': 'Contratações diversas',
    # Compras
    'Compras  [Outros custos internos]':               'Compras',
    'Compras':                                          'Compras',
    'Aquisição de material':                           'Compras',
    # Alimentação
    'Kit Lanche':                                       'Alimentação',
    'Brindes':                                          'Alimentação',
    'Café para integrações/Ações Similares':           'Alimentação',
    'Coffee break Tipo 1':                             'Alimentação',
    'Coffee break Tipo 2':                             'Alimentação',
    'Coquetel':                                         'Alimentação',
    'Recepções / Refeições':                           'Alimentação',
    'Diversos - Alimentação [Outros custos de terceiros]': 'Alimentação',
    # Infraestrutura de evento
    'Sonorização':                                      'Sonorização',
    'Locação - Sonorização [Outros custos de terceiros]': 'Sonorização',
    'Iluminação':                                       'Iluminação',
    'Locação - Iluminação [Outros custos de terceiros]': 'Iluminação',
    'Audiovisual / Projeção':                          'Audiovisual',
    'Audiovisual':                                      'Audiovisual',
    'Locação - Outros [Outros custos de terceiros]':   'Locação',
    'Locação':                                          'Locação',
    # Comunicação
    'Impressos e digitais':                            'Comunicação',
    'Editoria web':                                     'Comunicação',
    'Assessoria de imprensa':                          'Comunicação',
    'Diversos - Comunicação [Outros custos de terceiros]': 'Comunicação',
    # Acessibilidade
    'Tradução Simultânea':                             'Acessibilidade',
    # Outros (categorias absorvidas)
    'Serviço de Receptivo':                            'Outros',
    'Mobiliário':                                       'Outros',
    'Montagem':                                         'Outros',
    'Equipamentos':                                     'Outros',
    'Limpeza':                                          'Outros',
    'Elétrica':                                         'Outros',
    'Acompanhamento':                                   'Outros',
    'Outros':                                           'Outros',
    'Outros - Terceiros [Outros custos terceiros]':    'Outros',
    'Outros - Internos [Outros custos internos]':      'Outros',
}

# Fallback por prefixo — para itens com código de conta no nome (ex: 'Hospedagem [3920')
PREFIXOS_ITEM_CUSTO = [
    ('Hospedagem',           'Hospedagem'),
    ('Guia de turismo',      'Turismo'),
    ('Locação de Automóvel', 'Transporte'),
    ('Serviço de Montagem',  'Outros'),
    ('Serviços Gerais',      'Outros'),
]


def normalizar_item_custo(idc: str, item_grupo: str) -> str:
    """Retorna categoria normalizada; None = descartar."""
    idc_s   = str(idc).strip()
    grupo_l = str(item_grupo).strip().lower()
    # Estagiário → descarte
    if 'estagi' in grupo_l or 'estagi' in idc_s.lower():
        return None
    # item_grupo = 'acessibilidade' → Acessibilidade
    if grupo_l == 'acessibilidade':
        return 'Acessibilidade'
    # Match exato
    if idc_s in MAPA_ITEM_CUSTO:
        return MAPA_ITEM_CUSTO[idc_s]
    # Fallback por prefixo
    for prefixo, categoria in PREFIXOS_ITEM_CUSTO:
        if idc_s.startswith(prefixo):
            return categoria
    # Sem mapeamento — mantém o valor bruto
    return idc_s


In [ ]:
# Ordem de exibição dos grupos em item_desc
# 0 = Contratos  1 = Passagem Aérea  2 = Hospedagem  3 = demais (alfabético)
GRUPO_ITEM_DESC = {
    'Contrato PJ':          0,
    'Contrato PF':          0,
    'Contrato Cooperativa': 0,
    'Passagem Aérea':       1,
    'Hospedagem':           2,
}
SEP_ITEM_DESC = chr(9472) * 28   # ────────────────────────────


def build_solicitacoes_desc(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── item_de_custo bruto ───────────────────────────────────────────────
    grupo = df['item_grupo'].fillna('').astype(str).str.strip()
    nome  = df['nome_item'].fillna('').astype(str).str.strip()
    nome_ate_traco = nome.str.split('-').str[0].str.strip()
    usar_nome = (grupo == '') | (grupo.str.lower() == 'null')
    df['item_de_custo'] = np.where(usar_nome, nome_ate_traco, grupo)

    # ── descarte ─────────────────────────────────────────────────────────
    df = df[~df['item_de_custo'].isin(ITENS_DESCARTAR)].copy()

    # ── normalização ─────────────────────────────────────────────────────
    df['item_norm'] = df.apply(
        lambda r: normalizar_item_custo(r['item_de_custo'], r['item_grupo']),
        axis=1,
    )
    df = df[df['item_norm'].notna()].copy()

    # ── chave de ordenação: (grupo, item_norm alfabético, solicitacao_id) ─
    df['_grp'] = df['item_norm'].map(lambda x: GRUPO_ITEM_DESC.get(x, 3))

    # ── custo formatado ───────────────────────────────────────────────────
    df['custo_fmt'] = df['custo'].apply(
        lambda x: 'R$ ' + f'{x:,.0f}'.replace(',', '.')
    )

    # ── linha individual ──────────────────────────────────────────────────
    desc = df['descricao'].fillna('').astype(str).str.strip()
    df['linha'] = df['item_norm'] + ' — ' + desc + ' — ' + df['custo_fmt']

    # ── agrega por atividade_id com separadores entre grupos ─────────────
    def montar_desc(sub):
        sub = sub.sort_values(['_grp', 'item_norm', 'solicitacao_id'])
        linhas = []
        grp_atual = None
        for _, row in sub.iterrows():
            if grp_atual is not None and row['_grp'] != grp_atual:
                linhas.append(SEP_ITEM_DESC)
            linhas.append(row['linha'])
            grp_atual = row['_grp']
        return chr(10).join(linhas)

    def agrupar(sub_df, col_name: str) -> pd.DataFrame:
        if sub_df.empty:
            return pd.DataFrame(columns=['atividade_id', col_name])
        return (
            sub_df.groupby('atividade_id')
            .apply(montar_desc)
            .reset_index(name=col_name)
        )

    # ── item_desc completo ────────────────────────────────────────────────
    resultado = (
        df.groupby('atividade_id')
        .apply(montar_desc)
        .reset_index(name='item_desc')
    )

    # ── descrições por categoria ──────────────────────────────────────────
    resultado = (
        resultado
        .merge(agrupar(df[df['_grp'] == 0],                    'contratos_desc'), on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Passagem Aérea'], 'passagem_desc'),  on='atividade_id', how='left')
        .merge(agrupar(df[df['item_norm'] == 'Hospedagem'],     'hospedagem_desc'), on='atividade_id', how='left')
    )

    # ── flags 0/1 ─────────────────────────────────────────────────────────
    resultado['tem_contrato']   = resultado['contratos_desc'].notna().astype(int)
    resultado['tem_passagem']   = resultado['passagem_desc'].notna().astype(int)
    resultado['tem_hospedagem'] = resultado['hospedagem_desc'].notna().astype(int)

    for col in ['contratos_desc', 'passagem_desc', 'hospedagem_desc']:
        resultado[col] = resultado[col].fillna('')

    return resultado


solicitacoes_desc_df = build_solicitacoes_desc(raw_solicitacoes_df)

print(f'solicitacoes_desc_df: {solicitacoes_desc_df.shape}')
print(f'  tem_contrato=1:   {solicitacoes_desc_df["tem_contrato"].sum()}')
print(f'  tem_passagem=1:   {solicitacoes_desc_df["tem_passagem"].sum()}')
print(f'  tem_hospedagem=1: {solicitacoes_desc_df["tem_hospedagem"].sum()}')
print()
print('Exemplo (primeira linha de item_desc):')
print(solicitacoes_desc_df['item_desc'].iloc[0])

## 12. PCAP


In [ ]:
pcap_re       = r'PCAP[^0-9]*([0-9]{13})'    # com captura — para str.extract
pcap_re_check = r'PCAP[^0-9]*(?:[0-9]{13})'  # sem captura — para str.contains

# ── 1. busca em descricao (complemento) da solicitacao ───────────────────
sol_pcap = raw_solicitacoes_df[
    (raw_solicitacoes_df['area'] == 'Administrativo') &
    raw_solicitacoes_df['descricao'].str.contains(pcap_re_check, flags=re.IGNORECASE, na=False)
][['atividade_id', 'solicitacao_id', 'descricao']].copy()

sol_pcap['pcap_num'] = (
    sol_pcap['descricao']
    .str.extract(pcap_re, flags=re.IGNORECASE)[0]
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
sol_pcap = sol_pcap.drop(columns=['descricao']).drop_duplicates(subset=['solicitacao_id'])

# ── 2. busca nos campos da justificativa (nivel atividade_id) ─────────────
_txt_cols_pcap = [c for c in ['sinopse_curta', 'sinopse_aprovacao', 'info_parceria', 'just_recursos']
                  if c in raw_acoes_txts_df.columns]

_txts = (
    raw_acoes_txts_df[['acao.atividade_id'] + _txt_cols_pcap]
    .rename(columns={'acao.atividade_id': 'atividade_id'})
    .drop_duplicates(subset=['atividade_id'])
    .copy()
)
_txts['_texto'] = _txts[_txt_cols_pcap].fillna('').apply(' '.join, axis=1)

just_pcap = _txts[
    _txts['_texto'].str.contains(pcap_re_check, flags=re.IGNORECASE, na=False)
][['atividade_id', '_texto']].copy()

just_pcap['pcap_num'] = (
    just_pcap['_texto']
    .str.extract(pcap_re, flags=re.IGNORECASE)[0]
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Int64')
)
just_pcap = just_pcap.drop(columns=['_texto']).dropna(subset=['pcap_num'])

# Exclui atividades ja encontradas via descricao
just_pcap = just_pcap[~just_pcap['atividade_id'].isin(set(sol_pcap['atividade_id']))]

# Associa a melhor solicitacao_id disponivel: prefere area Administrativo
_solic_best = (
    pd.concat([
        raw_solicitacoes_df[raw_solicitacoes_df['area'] == 'Administrativo'][['atividade_id', 'solicitacao_id']].assign(_prio=0),
        raw_solicitacoes_df[['atividade_id', 'solicitacao_id']].assign(_prio=1),
    ])
    .sort_values(['atividade_id', '_prio'])
    .drop_duplicates(subset=['atividade_id'])
    [['atividade_id', 'solicitacao_id']]
)
just_pcap = just_pcap.merge(_solic_best, on='atividade_id', how='left')

print(f'PCAP via complemento:   {sol_pcap["atividade_id"].nunique()} atividades')
print(f'PCAP via justificativa: {just_pcap["atividade_id"].nunique()} atividades (novas)')

# ── 3. une as duas fontes ──────────────────────────────────────────────────
sol_pcap = pd.concat([sol_pcap, just_pcap], ignore_index=True).drop_duplicates(subset=['solicitacao_id'])

raw_pcap_df = sol_pcap.merge(pcap_props_df, on='pcap_num', how='inner')
print(f'raw_pcap_df:       {raw_pcap_df.shape}')
print(f'atividades unicas: {raw_pcap_df["atividade_id"].nunique()}')

In [ ]:
# Detalhamento por grupo_item (texto para exibição ao usuário)
_det = (
    raw_pcac_det_df
    .groupby(['id_proposta', 'grupo_item'], as_index=False)['valor_total_item']
    .sum()
    .sort_values(['id_proposta', 'valor_total_item'], ascending=[True, False])
)

def _fmt_brl(v):
    return f'R$ {v:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

_det_grupos = (
    _det
    .groupby('id_proposta')
    .apply(lambda g: '\n'.join(
        f"{row['grupo_item']}: {_fmt_brl(row['valor_total_item'])}"
        for _, row in g.iterrows()
    ))
    .reset_index(name='det_grupos')
)

raw_pcap_df = raw_pcap_df.merge(_det_grupos, on='id_proposta', how='left')
print(f'raw_pcap_df com det_grupos: {raw_pcap_df.shape}')

In [ ]:
autonomias_df = build_autonomias(rps_parcial_df, datas_df, contracts_df, raw_pcap_df)
autonomias_df['autonomia'].value_counts()

## 13. Tabela Base


In [ ]:
SUBATIV_PERMANENTE = frozenset({
    'Acesso a recursos informacionais',
    'Análise de risco em saúde',
    'Consulta',
    'Exercício físicos sistematicos',
    'Formação esportiva',
    'Sessão diagnóstica/clínica',
    'Refeição',
    'Lanche',
    'Capacitação e Desenvolvimento de Empregados',
    'Rádio e TV',
    'Colônias recreativas',
    'Relacionamento com clientes',
    'Distribuição de doações',
    'Creche',
    'Pré-escola',
    'Acesso a recursos informacionais',
    'Procedimentos clínicos',
    'Procedimentos complementares',
    'Práticas coletivas',
    'Parque aquático',
    'Hospedagem',
})

SUBATIV_EVENTUAL = frozenset({
    'Apresentação',
    'Competições físico-esportivas',
    'Multipráticas recreativas',
    'Incentivo artístico e cultural',
    'Eventos',
    'Exibição',
    'Exposição',
    'Passeios',
    'Produtos gastronômicos',
    'Viagens',
})


def build_tabela_base(
    raw_acoes_df, datas_df, autonomias_df, raw_projetos_df,
    todas_as_datas_df, raw_tags_df, solicitacoes_desc_df, contracts_df,
    raw_acessibilidade_df, raw_pcap_df, raw_datas_sessoes_df,
    precif_df,
    justificativa_df,
) -> pd.DataFrame:

    _acoes_drop = ['projeto'] + [c for c in raw_acoes_df.columns if c == 'primeiradata' or c.endswith('.primeiradata')]
    df = raw_acoes_df.drop(columns=_acoes_drop, errors='ignore').copy()

    # ── faixa etária ──────────────────────────────────────────────────────
    df['a.idade_inicial'] = pd.to_numeric(df['a.idade_inicial'], errors='coerce').fillna(0)
    df['a.idade_final']   = pd.to_numeric(df['a.idade_final'],   errors='coerce').fillna(0)
    temp0 = df['a.idade_inicial'] + df['a.idade_final']
    temp1 = np.select(
        [df['a.linguagem'] == 'Crianças',
         df['a.linguagem'] == 'Idosos',
         df['a.recomendacao_etaria'] != 'Livre'],
        ['infantil', 'pessoas idosas', 'não é'],
        default='pode ser',
    )
    temp2 = np.select(
        [temp0 == 0, df['a.idade_inicial'] >= 60, df['a.idade_inicial'] >= 12,
         df['a.idade_final'] == 0, df['a.idade_final'] <= 13, df['a.idade_final'] <= 15],
        ['s/i', 'idosos', 'não é', 's/i', 'infantil', 'pode ser'],
        default='não é',
    )
    mesclado = pd.Series(np.array(temp1, dtype=object) + ' - ' + np.array(temp2, dtype=object), index=df.index)
    df['faixa'] = np.select(
        [mesclado.str.contains('infantil'), mesclado.str.contains('idosos'),
         mesclado.str.contains('pode ser'), mesclado.str.contains('s/i')],
        ['infantil', 'pessoas idosas', 's/i', 's/i'],
        default='s/i',
    )

    # ── joins 1:1 ─────────────────────────────────────────────────────────
    datas_cols = [
        'atividade_id', 'PrimeiraData', 'PrimeiraHora',
        'ultimadata', 'qt_sessoes', 'qt_datas_distintas', 'qt_horas',
        'tempo_da_sessao', 'diascorridos', 'mes',
        '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
    ]
    df = df.merge(
        datas_df[[c for c in datas_cols if c in datas_df.columns]],
        on='atividade_id', how='left',
    )
    df = df.merge(
        autonomias_df[['atividade_id', 'autonomia', 'autonomiaTemporal', 'autonomiaCusto', 'autonomiaPCAP']],
        on='atividade_id', how='left',
    )
    df = df.merge(
        raw_projetos_df.drop(columns=['institucional'], errors='ignore'),
        on='projeto_id', how='left',
    )
    df = df.merge(todas_as_datas_df, on='atividade_id', how='left')
    df = df.merge(
        raw_tags_df.drop_duplicates('atividade_id')[['atividade_id', 'todas_as_tags']],
        on='atividade_id', how='left',
    )
    df = df.merge(solicitacoes_desc_df, on='atividade_id', how='left')
    df = df.merge(
        contracts_df.drop_duplicates('atividade_id')[
            ['atividade_id', 'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic']
        ],
        on='atividade_id', how='left',
    )
    df = df.merge(
        precif_df[['atividade_id', 'gratuito', 'maior_valor', 'menor_valor']],
        on='atividade_id', how='left',
    )

    # ── periodicidade ─────────────────────────────────────────────────────
    # Regras (mutuamente exclusivas):
    # 1. Subatividades sempre permanentes
    # 2. Subatividades sempre eventuais (Ações formativas/mediadas excluem Curso/Vivência)
    # 3. Curso (subativ=Ações formativas) e Vivência (subativ=Ações mediadas):
    #    diascorridos > 90 E qt_sessoes > 30 → permanente; caso contrário → eventual
    dias = pd.to_numeric(df['diascorridos'], errors='coerce').fillna(0)
    sess = pd.to_numeric(df['qt_sessoes'],   errors='coerce').fillna(0)

    cond_perm  = df['subatividade'].isin(SUBATIV_PERMANENTE)
    cond_ev    = (
        df['subatividade'].isin(SUBATIV_EVENTUAL) |
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] != 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] != 'Vivência'))
    )
    cond_cv    = (
        ((df['subatividade'] == 'Ações formativas') & (df['servico'] == 'Curso')) |
        ((df['subatividade'] == 'Ações mediadas')   & (df['servico'] == 'Vivência'))
    )
    cv_perm = cond_cv & (dias > 90) & (sess > 30)
    cv_ev   = cond_cv & ~((dias > 90) & (sess > 30))

    df['periodicidade'] = np.select(
        [cond_perm, cond_ev, cv_perm, cv_ev],
        ['permanente', 'eventual', 'permanente', 'eventual'],
        default='s/i',
    )

    # Permanente sem contrato → autonomia UO (independente das outras fontes)
    _sem_contrato = df['custo_contratos_total'].fillna(0) == 0
    df.loc[(df['periodicidade'] == 'permanente') & _sem_contrato, 'autonomia'] = 'UO'

    # ── flags via .isin() ─────────────────────────────────────────────────
    df['tem_dispositivo'] = df['atividade_id'].isin(raw_acessibilidade_df['atividade_id']).astype(int)
    _acess_agg = (
        raw_acessibilidade_df[['atividade_id', 'a.identificacao']]
        .dropna(subset=['a.identificacao'])
        .groupby('atividade_id')
        .agg(
            todas_os_dispositivos=('a.identificacao', lambda x: ' | '.join(sorted(x.unique()))),
            n_dispositivos=('a.identificacao', 'nunique'),
        )
        .reset_index()
    )
    df = df.merge(_acess_agg, on='atividade_id', how='left')
    df['todas_os_dispositivos'] = df['todas_os_dispositivos'].fillna('')
    df['n_dispositivos'] = df['n_dispositivos'].fillna(0).astype(int)
    df['com_pcap']        = df['atividade_id'].isin(raw_pcap_df['atividade_id']).astype(int)

    # ── espaco_brincar: OR de quatro fontes ───────────────────────────────
    _eb = set().union(
        raw_acoes_df.loc[raw_acoes_df['a.nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_tags_df.loc[raw_tags_df['tag_nome'].str.contains('Espaço de Brincar', na=False), 'atividade_id'],
        raw_acoes_df.loc[raw_acoes_df['projeto_id'].isin(set(
            raw_projetos_df.loc[
                raw_projetos_df['projeto_uo_nome'].str.contains('Espaço de Brincar', na=False) |
                raw_projetos_df['tag_projeto'].str.contains('Espaço de Brincar', na=False),
                'projeto_id',
            ]
        )), 'atividade_id'],
        raw_datas_sessoes_df.loc[
            raw_datas_sessoes_df['localNome'].str.contains('Espaço de Brincar', na=False) |
            raw_datas_sessoes_df['TipologiaLocal'].str.contains('Espaço de Brincar', na=False),
            'atividade_id',
        ],
    )
    df['espaco_brincar'] = df['atividade_id'].isin(_eb).astype(int)

    # ── integra_expo: exposições longas (>30 sessões) com projeto → propaga a flag para todo o projeto
    _projetos_expo = set(
        df.loc[
            (df['servico'] == 'Exposição') &
            (sess > 30) &
            df['projeto_id'].notna() &
            (df['projeto_id'] != ''),
            'projeto_id',
        ]
    )
    df['integra_expo'] = (
        df['projeto_id'].isin(_projetos_expo) & df['projeto_id'].notna()
    ).astype(int)

    # ── periodicidade: refinamentos pós-flags ─────────────────────────────────
    # Vivência + espaco_brincar + sessoes > 18 → permanente
    _viv_eb = (
        (df['servico'] == 'Vivência')
        & (sess > 18)
        & (df['espaco_brincar'] == 1)
    )
    # Curso + sessoes > 18 + Curumim/Juventudes/Centro de Música em nome, projeto_nome ou complemento
    _RE_PERM_CURSO = r'curumim|juventudes?|centro de m[uú]sica'
    _nome_proj = (
        df['a.nome'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
        | df['projeto_nome'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
        | df['a.complemento'].fillna('').str.contains(_RE_PERM_CURSO, case=False)
    )
    _curso_prog = (df['servico'] == 'Curso') & (sess > 18) & _nome_proj
    df.loc[_viv_eb | _curso_prog, 'periodicidade'] = 'permanente'
    df.loc[(_viv_eb | _curso_prog) & _sem_contrato, 'autonomia'] = 'UO'

    # ── tem_educador: educadores no campo complemento ─────────────────────────
    # tem_educador=1: educador+contexto (ou agente ambiental) com custo_contratos_total=0
    # tem_educador=2: idem com custo_contratos_total>0
    _comp = df['a.complemento'].fillna('')
    _RE_CTX = (
        r'tecnologia[s]? e arte[s]?'
        r'|\bsesc\b'
        r'|\beta\b'
        r'|da exposi[cç][aã]o'
        r'|f[ií]sico[- ]esportiv[ao]s?'
        r'|infanto[- ]juveni[ls]'
        r'|curumim'
        r'|juventudes?'
        r'|espa[cç]o de brincar'
        r'|centro de m[uú]sica'
        r'|da unidade'
        r'|s[oó]cio[- ]?educativ[ao]s?'
        r'|agente de educa[cç][aã]o ambiental'
        r'|atividades? musicais?'
    )
    _has_educ_ctx = (
        _comp.str.contains(r'educador[ae]?s?', case=False)
        & _comp.str.contains(_RE_CTX, case=False)
    ) | _comp.str.contains(r'agente de educa[cç][aã]o ambiental', case=False)
    _custo_educ = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0)
    df['tem_educador'] = np.where(
        _has_educ_ctx & (_custo_educ == 0), 1,
        np.where(_has_educ_ctx & (_custo_educ > 0), 2, 0)
    ).astype(int)

    # tem_educador=1 -> autonomia UO independente de dias corridos e carga horária
    df.loc[df['tem_educador'] == 1, 'autonomia'] = 'UO'

    # ── reordenação de colunas ────────────────────────────────────────────
    df = df.merge(justificativa_df[['atividade_id', 'justificativa']], on='atividade_id', how='left')

    col_order = [
        # Identificação
        'uo', 'atividade_id', 'status_atividade', 'nome', 'a.complemento',
        # Hierarquia programática
        'areaprog', 'atividade', 'subatividade', 'servico', 'periodicidade',
        # Classificação
        'tipo', 'subtipo', 'formato', 'linguagem',
        # Público e faixa etária
        'recomendacao_etaria', 'faixa', 'estimativa_publico', 'lugares',
        # Datas e sessões
        'PrimeiraData', 'PrimeiraHora', 'ultimadata',
        'qt_sessoes', 'qt_datas_distintas', 'qt_horas', 'tempo_da_sessao',
        'diascorridos', 'mes', '30dias', '90dias', '60horas', 'ExtrapolaDataHora',
        # Flags de ano
        'ano_2018', 'ano_2019', 'ano_2020', 'ano_2021', 'ano_2022',
        'ano_2023', 'ano_2024', 'ano_2025', 'ano_2026',
        # Texto de datas
        'todas_as_datas',
        # Autonomia
        'autonomia', 'autonomiaTemporal', 'autonomiaCusto', 'autonomiaPCAP',
        # Projeto
        'projeto_id', 'projeto_nome', 'projeto_complemento', 'projeto_categoria',
        'tag_projeto', 'tag_grupo_projeto', 'projeto_uo_nome',
        'projeto_descricao', 'projeto_comunicacao', 'projeto_conceitual', 'tem_pai',
        'justificativa',
        # Tags
        'todas_as_tags',
        # Custos — flags e totais
        'tem_contrato', 'tem_passagem', 'tem_hospedagem',
        'custo_contratos_total', 'custo_total', 'n_contratos', 'n_solic',
        # Custos — descrições
        'item_desc', 'contratos_desc', 'passagem_desc', 'hospedagem_desc',
        # Flags especiais
        'tem_dispositivo', 'todas_os_dispositivos', 'n_dispositivos', 'espaco_brincar', 'integra_expo', 'com_pcap', 'tem_educador',
        # Complementares
        'gratuito', 'maior_valor', 'menor_valor', 'precificacao_desc', 'produtor', 'tem_parceria',
        'contatofornecedores', 'uso_interno', 'manutencao', 'regular',
        'integracao_sgc', 'institucional',
    ]
    ordered = [c for c in col_order if c in df.columns]
    extras  = [c for c in df.columns if c not in ordered]
    return df[ordered + extras]

In [ ]:
# Justificativa: concatena textos de raw_acoes_txts separados por linha em branco
_txt_cols = ['sinopse_curta', 'sinopse_aprovacao', 'info_parceria', 'just_recursos']

justificativa_df = (
    raw_acoes_txts_df[['acao.atividade_id'] + [c for c in _txt_cols if c in raw_acoes_txts_df.columns]]
    .rename(columns={'acao.atividade_id': 'atividade_id'})
    .drop_duplicates(subset=['atividade_id'])
    .copy()
)

def _join_partes(row):
    parts = []
    for c in _txt_cols:
        if c in row.index:
            val = '' if pd.isna(row[c]) else str(row[c]).strip()
            if val:
                parts.append(val)
    return '\n\n'.join(parts)

justificativa_df['justificativa'] = justificativa_df.apply(_join_partes, axis=1)
justificativa_df = justificativa_df[['atividade_id', 'justificativa']]
print(f'justificativa_df: {justificativa_df.shape}')

In [ ]:
tabela_base_df = build_tabela_base(
    raw_acoes_df        = raw_acoes_df,
    datas_df            = datas_df,
    autonomias_df       = autonomias_df,
    raw_projetos_df     = raw_projetos_df,
    todas_as_datas_df   = todas_as_datas_df,
    raw_tags_df         = raw_tags_df,
    solicitacoes_desc_df= solicitacoes_desc_df,
    contracts_df        = contracts_df,
    raw_acessibilidade_df = raw_acessibilidade_df,
    raw_pcap_df         = raw_pcap_df,
    raw_datas_sessoes_df= raw_datas_sessoes_df,
    precif_df           = precif_df,
    justificativa_df    = justificativa_df,
)

print(f'tabela_base_df: {tabela_base_df.shape}')
print(f'atividade_id única: {tabela_base_df["atividade_id"].is_unique}')
print()
print('faixa:')
print(tabela_base_df['faixa'].value_counts())
print()
print('autonomia:')
print(tabela_base_df['autonomia'].value_counts())
print()
flags = ['tem_dispositivo', 'espaco_brincar', 'integra_expo', 'com_pcap',
         'tem_contrato', 'tem_passagem', 'tem_hospedagem']
for f in flags:
    if f in tabela_base_df.columns:
        print(f'{f}=1: {tabela_base_df[f].sum()}')

### Verifica se os joins geraram mais de uma linha

In [ ]:
# Isola as linhas duplicadas
raw_acoes_orig = read_raw(f'raw_acoes{_all}')
raw_acoes_orig['atividade_id'] = raw_acoes_orig['atividade_id'].astype(str).str.strip()

mask_dup = raw_acoes_orig.duplicated(subset=['atividade_id'], keep=False)
dupes = raw_acoes_orig[mask_dup].sort_values('atividade_id')

# Quais colunas variam entre as cópias de uma mesma atividade_id?
cols_variam = [
    col for col in dupes.columns
    if dupes.groupby('atividade_id')[col].nunique().max() > 1
]
print('Colunas com valores distintos entre duplicatas:')
print(cols_variam)

# Exemplo de uma atividade duplicada
exemplo = dupes['atividade_id'].iloc[0]
print(f'\nExemplo — atividade_id {exemplo}:')
dupes[dupes['atividade_id'] == exemplo][['atividade_id'] + cols_variam]


## Define as gerências

In [ ]:
# ── Gerências (tabela_base) ────────────────────────────────────────────────
#
# Três colunas derivadas:
#   gerencia  — gerência principal mapeada 1:1 de areaprog
#   gerenciaB — gerência secundária; regras baseadas em atividade / solicitações
#   gerencias — combinação pipe-sep das duas (sem pipe se igual; nulo se ambas nulas)
#
# ── Como adicionar novas regras em gerenciaB ──────────────────────────────
# 1. Crie um set/frozenset ou uma máscara booleana que identifique as atividades.
# 2. Adicione um bloco ".loc[mask, 'gerenciaB'] = 'SIGLA'" abaixo dos existentes,
#    ANTES das regras de prioridade mais alta (última linha escrita prevalece).
#    Exemplo: para nova regra com prioridade entre GESC e GEAVT, insira após GESC.
# ─────────────────────────────────────────────────────────────────────────

# ── gerencia ──────────────────────────────────────────────────────────────
GERENCIA_POR_AREAPROG = {
    'Gestão Cultural e Esportiva':        'GEDES-CPF',
    'Credenciamento':                     'GEARP',
    'Bem Viver':                          'GEDEP',
    'Conteúdo em Mídias':                 'GSD',
    'Eventos Físico-Esportivos':          'GDFE',
    'Programa de Ginástica Multifuncional': 'GDFE',
    'Programa de Práticas Aquáticas':     'GDFE',
    'Programa de Práticas Corporais':     'GDFE',
    'Programa Sesc de Esportes':          'GDFE',
    'Audiovisual':                        'GEAC',
    'Biblioteca':                         'GEAC',
    'Circo':                              'GEAC',
    'Dança':                              'GEAC',
    'Literatura':                         'GEAC',
    'Música':                             'GEAC',
    'Teatro':                             'GEAC',
    'Alimentação - Ações educativas':     'GEASA',
    'Sesc Mesa Brasil':                   'GEASA',
    'Alimentação':                        'GEASA',
    'Artes Visuais':                      'GEAVT',
    'Tecnologias e Artes':                'GEAVT',
    'Direitos humanos':                   'GEPROS',
    'Gênero e Sexualidade':               'GEPROS',
    'Infâncias':                          'GEPROS',
    'Juventudes':                         'GEPROS',
    'Negritude':                          'GEPROS',
    'Povos e Comunidades Tradicionais':   'GEPROS',
    'Povos Indígenas':                    'GEPROS',
    'Refúgio e Migração':                 'GEPROS',
    'Trabalho Social com Pessoas Idosas': 'GEPROS',
    'CEDEI':                              'GEPROS',
    'Desenvolvimento Comunitário':        'GESC',
    'Educação para Acessibilidade':       'GESC',
    'Educação para Sustentabilidade':     'GESC',
    'Turismo Social':                     'GESC',
    'Valorização Social':                 'GESC',
    'Qualidade de Vida':                  'GSO',
    'Saúde Bucal':                        'GSO',
    'Saúde Mental':                       'GSO',
    'Saúde Sexual e Reprodutiva':         'GSO',
}

tabela_base_df['gerencia'] = tabela_base_df['areaprog'].map(GERENCIA_POR_AREAPROG)

# ── gerenciaB ─────────────────────────────────────────────────────────────
# Regras avaliadas em ordem de prioridade crescente (última escrita prevalece).
# Prioridade: GEAC > GEAVT > GESC

# Regra 3 (prioridade baixa): solicitação com item_grupo='Acessibilidade' e custo > 0 → GESC
# Busca em raw_solicitacoes_df, que inclui todos os tipos de item (não só contratos).
ATIVIDADES_GERENB_GEAC = frozenset({
    'Artes Cênicas - Circo',
    'Artes Cênicas - Dança',
    'Artes Cênicas - Teatro',
    'Audiovisual',
    'Biblioteca',
    'Literatura',
    'Música',
})

# Regra 2: atividade = 'Artes Visuais' → GEAVT
ATIVIDADES_GERENB_GEAVT = frozenset({'Artes Visuais'})

_ids_acess_gesc = set(
    raw_solicitacoes_df.loc[
        (raw_solicitacoes_df['item_grupo'].astype(str).str.strip() == 'Acessibilidade') &
        (raw_solicitacoes_df['custo'] > 0),
        'atividade_id',
    ].unique()
)

_ativ = tabela_base_df['atividade'].fillna('').str.strip()

tabela_base_df['gerenciaB'] = None
# Aplica da prioridade mais baixa para a mais alta (última escrita vence):
tabela_base_df.loc[tabela_base_df['atividade_id'].isin(_ids_acess_gesc), 'gerenciaB'] = 'GESC'
tabela_base_df.loc[_ativ.isin(ATIVIDADES_GERENB_GEAVT), 'gerenciaB'] = 'GEAVT'
tabela_base_df.loc[_ativ.isin(ATIVIDADES_GERENB_GEAC),  'gerenciaB'] = 'GEAC'

# ── gerencias ─────────────────────────────────────────────────────────────
_g1 = tabela_base_df['gerencia'].fillna('')
_g2 = tabela_base_df['gerenciaB'].fillna('')

_so_g1   = (_g1 != '') & (_g2 == '')
_so_g2   = (_g1 == '') & (_g2 != '')
_iguais  = (_g1 != '') & (_g2 != '') & (_g1 == _g2)
_diferen = (_g1 != '') & (_g2 != '') & (_g1 != _g2)

tabela_base_df['gerencias'] = None
tabela_base_df.loc[_so_g1,   'gerencias'] = _g1[_so_g1]
tabela_base_df.loc[_so_g2,   'gerencias'] = _g2[_so_g2]
tabela_base_df.loc[_iguais,  'gerencias'] = _g1[_iguais]
tabela_base_df.loc[_diferen, 'gerencias'] = (_g1 + '|' + _g2)[_diferen]

# ── diagnóstico ───────────────────────────────────────────────────────────
print('gerencia:')
print(tabela_base_df['gerencia'].value_counts(dropna=False))
print()
print(f'gerenciaB (ids com Acessibilidade: {len(_ids_acess_gesc)}):')
print(tabela_base_df['gerenciaB'].value_counts(dropna=False))
print()
print('gerencias (top 15):')
print(tabela_base_df['gerencias'].value_counts(dropna=False).head(15))

In [ ]:
# ── Ponte gerência × ação (many-to-many para Power BI) ────────────────────
#
# dim_gerencia  : dimensão com as siglas únicas
# ponte_gerencia: bridge table atividade_id × sigla (uma linha por par)
#
# Permite filtrar tabela_base por gerência incluindo ações compartilhadas
# — onde a gerência do usuário aparece como principal OU secundária.

dim_gerencia_df = pd.DataFrame(
    {'sigla': sorted(set(GERENCIA_POR_AREAPROG.values()))}
)

ponte_gerencia_df = (
    tabela_base_df[['atividade_id', 'gerencias']]
    .dropna(subset=['gerencias'])
    .assign(sigla=lambda df: df['gerencias'].str.split('|'))
    .explode('sigla')
    .assign(sigla=lambda df: df['sigla'].str.strip())
    .loc[lambda df: df['sigla'] != '']
    [['atividade_id', 'sigla']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f'dim_gerencia: {dim_gerencia_df.shape[0]} siglas')
print(f'ponte_gerencia: {ponte_gerencia_df.shape[0]} pares atividade x gerencia')
print(ponte_gerencia_df['sigla'].value_counts())

## Limpa as colunas antes de salvar
- data e hora
- normaliza os nomes das colunas eliminando sujeiras antes do ponto
- transforma atividade_id e sessao_id em numérico decimal

In [ ]:
import datetime

def sanitize_for_spark(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # Remove prefixo de alias SQL (ex.: "a.complemento" → "complemento")
    df.columns = [col.split('.')[-1] for col in df.columns]

    df = df.loc[:, ~df.columns.duplicated()]

    # atividade_id e sessao_id como numérico decimal
    for id_col in ['atividade_id', 'sessao_id']:
        if id_col in df.columns:
            df[id_col] = pd.to_numeric(df[id_col], errors='coerce')

    for col in df.columns:

        # timedelta → segundos inteiros (DayTimeIntervalType não é suportado pelo Delta)
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col] = df[col].dt.total_seconds().astype('Int64')
            continue

        amostra = df[col].dropna()
        if not len(amostra):
            continue
        sample = amostra.iloc[0]

        if isinstance(sample, datetime.time):
            # datetime.time → string; Spark/Arrow não suportam time64
            df[col] = df[col].apply(
                lambda t: t.strftime('%H:%M:%S') if isinstance(t, datetime.time) else None
            )
        elif isinstance(sample, (datetime.date, datetime.datetime)):
            df[col] = pd.to_datetime(df[col], errors='coerce')

    return df



def save_gold(df: pd.DataFrame, table_name: str) -> None:
    spark.createDataFrame(sanitize_for_spark(df)) \
         .write.mode('overwrite') \
         .option('overwriteSchema', 'true') \
         .saveAsTable(f'lake_gold_fatos.dbo.{table_name}')

### Transforma tags em áreas programáticas para filtragem expandida


In [ ]:
# area_prog_tag: atividade_id + areaprogTag + origem

# -- Constantes de exclusão --
_EXCLUIR_GRUPO = {
    '', 'Abordagem (Projeto)', 'Acervo Sesc (Projeto)', 'Acao Online',
    'Bem Viver', 'Conteudos (Projeto)', 'Desenvolvimento de pessoal',
    'Editorial', 'Educacao formal', 'Familias', 'Formato (Projeto)',
    'Geral', 'Performance', 'Procedencia (Projeto)', 'Recorte Geografico (Projeto)',
    'Ação Online', 'Conteúdos (Projeto)', 'Famílias', 'Procedência (Projeto)',
    'Recorte Geográfico (Projeto)', 'Educação formal',
}
_EXCLUIR_FINAL = {
    'Caravana', 'Circuito Sesc', 'Circuito', 'Desenvolvimento de habilidades ',
    'destaques 2022', 'Destaques 2023', 'Destaques', 'Difusão de saberes',
    'Filosofia e Ciências Sociais', 'Mobilização - Plena Pausa', 'Mobilização',
    'Seminários', 'Sesc Gerações',
}

# -- Funções auxiliares --
def _calc_areaprog_tag(r):
    g, n = r['tag_grupo'], str(r['tag_nome'])
    if g == 'Diversidade Cultural': return n
    if g == 'Infâncias e Juventudes': return n
    if 'Mesa Brasil' in n: return 'Sesc Mesa Brasil'
    if g == 'Educação em Saúde': return n
    return g

def _calc_area_com_tag(r):
    n, g = str(r['tag_nome']), str(r['tag_grupo'])
    if n == 'Programa Sesc de Esportes': return 'Programa Sesc de Esportes'
    if 'Multifuncional' in n: return 'Programa de Ginástica Multifuncional'
    if 'Aquáticas' in n: return 'Programa de Atividades Aquáticas'
    if 'Corporais' in n: return 'Programa de Práticas Corporais'
    if 'Esporte' in g: return 'Eventos Físico-Esportivos'
    return 'areaprog_tag'

def _transform_tags(df):
    """Pipeline: df(atividade_id, tag_grupo, tag_nome) → df(atividade_id, areaprogTag)."""
    df = df[~df['tag_grupo'].fillna('').isin(_EXCLUIR_GRUPO)].copy()
    df['areaprog_tag'] = df.apply(_calc_areaprog_tag, axis=1)
    df['areaprog_tag'] = df['areaprog_tag'].str.replace('Do 13 ao 20', 'Negritude', regex=False)
    df = df[df['areaprog_tag'] != 'Diversidade Cultural']

    for _o, _n in [
        ('Crianças', 'Infâncias'), ('Bebês', 'Infâncias'), ('Espaço de Brincar', 'Infâncias'),
        ('Legítima Diferença', 'Gênero e Sexualidade'), ('Agosto Indígena', 'Povos Indígenas'),
        ('Jovens', 'Juventudes'), ('Adolescentes', 'Juventudes'), ('Abril Indígena', 'Povos Indígenas'),
        ('Culturas em Trânsito', 'Refúgio e Migração'), ('Semana Mundial do Brincar', 'Infâncias'),
        ('Coisa de Criança', 'Infâncias'),
    ]:
        df['areaprog_tag'] = df['areaprog_tag'].str.replace(_o, _n, regex=False)

    df['_act'] = df.apply(_calc_area_com_tag, axis=1)
    df['area_com_tag2'] = np.where(df['_act'] == 'areaprog_tag', df['areaprog_tag'], df['_act'])

    for _o, _n in [
        ('Educação em Saúde', 'Qualidade de Vida'), ('Nas Férias...', 'Infâncias'),
        ('Nutrição', 'Alimentação'), (' Urbanismo e Design', 'Tecnologias e Artes'),
        ('Arquitetura,Tecnologias e Artes', 'Tecnologias e Artes'),
        ('Audiovisual e Produção Sonora', 'Tecnologias e Artes'),
        (' Games e Cultura Geek', 'Tecnologias e Artes'), (' Eletrônica e Hardware', 'Tecnologias e Artes'),
        ('Ciência e Tecnologias', 'Tecnologias e Artes'), ('Sustentabilidade', 'Educação para Sustentabilidade'),
        ('Gestão Cultural', 'Gestão Cultural e Esportiva'), (' Costura e Moda', 'Tecnologias e Artes'),
        ('Curumim35anos', 'Infâncias'), (' Artesanato e Craft', 'Tecnologias e Artes'),
    ]:
        df['area_com_tag2'] = df['area_com_tag2'].str.replace(_o, _n, regex=False)

    for _o, _n in [
        ('Educação em Saúde', 'Qualidade de Vida'), ('Saúde', 'Qualidade de Vida'),
        ('Atividades formativas em gestão e mediação culturais', 'Gestão Cultural e Esportiva'),
        ('Acompanhamento GDFE', 'Eventos Físico-Esportivos'),
        ('Letramento e Inclusão Digital', 'Tecnologias e Artes'),
    ]:
        df['tag_nome'] = df['tag_nome'].astype(str).str.replace(_o, _n, regex=False)

    df['areaprogTag'] = np.where(df['area_com_tag2'] == 'null', df['tag_nome'], df['area_com_tag2'])
    df = df[['atividade_id', 'areaprogTag']].drop_duplicates()
    df = df[df['areaprogTag'].notna() & (df['areaprogTag'] != 'null') & (df['areaprogTag'].str.strip() != '')]

    for _o, _n in [
        ('Acessibilidade', 'Educação para Acessibilidade'),
        ('Cursos e Oficinas Artes Visuais', 'Tecnologias e Artes'),
        ('Curumim', 'Infâncias'),
        ('Educação para Educação para Sustentabilidade', 'Educação para Sustentabilidade'),
        ('Qualidade de vida', 'Qualidade de Vida'),
        ('Artesanias e ofícios tradicionais', 'Tecnologias e Artes'),
        ('Acompanhamento GDFE', 'Eventos Físico-Esportivos'),
        ('Trabalho Social com Pessoas Idosas', 'Pessoas Idosas'),
        ('Esporte e Atividade Física', 'Eventos Físico-Esportivos'),
        ('MPB', 'Música'),
        ('Arte têxtil e moda', 'Tecnologias e Artes'),
        ('Saúde mental e emocional [BV]', 'Bem Viver'),
        ('Audiovisual/Imagem e som', 'Audiovisual'),
        ('Cinema e audiovisual', 'Audiovisual'),
        ('Documentário', 'Audiovisual'),
        ('Cinema', 'Audiovisual'),
        ('Adulto - Ator', 'Teatro'),
        ('Adulto - Ator', 'Teatro'),
        ('Artes Visuais e gráficas', 'Artes Visuais'),
    ]:
        df['areaprogTag'] = df['areaprogTag'].str.replace(_o, _n, regex=False)

    df = df[~df['areaprogTag'].isin(_EXCLUIR_FINAL)]

    for _o, _n in [('PICS', 'Qualidade de Vida'), ('Gordofobia', 'Qualidade de Vida')]:
        df['areaprogTag'] = df['areaprogTag'].str.replace(_o, _n, regex=False)

    return df

# -- Parte 1: areaprog direto --
_ap_col = 'areaprog' if 'areaprog' in raw_acoes_df.columns else 'a.areaprog'
_pt1 = (
    raw_acoes_df[['atividade_id', _ap_col]]
    .rename(columns={_ap_col: 'areaprogTag'})
    .dropna(subset=['areaprogTag'])
    .copy()
)
_pt1 = _pt1[_pt1['areaprogTag'].str.strip().ne('') & _pt1['areaprogTag'].ne('null')]
_pt1['origem'] = 'areaprog'
_pt1 = _pt1.drop_duplicates(subset=['atividade_id', 'areaprogTag'])

# -- Parte 2: tags (raw_tags_df, já explodido) --
_t = raw_tags_df[['atividade_id', 'tag_nome', 'tag_grupo']].copy()
_t = _t[_t['atividade_id'].isin(raw_acoes_df['atividade_id'])]
_t = _transform_tags(_t)
_t['origem'] = 'tag'

# -- Parte 3: tags de projeto (pipe-separated em raw_projetos_df, join por projeto_id) --
_gp_col = next((c for c in raw_projetos_df.columns if 'tag_grupo_projeto' in c), None)
_tp_col = next((c for c in raw_projetos_df.columns if c == 'tag_projeto' or c.endswith('.tag_projeto')), None)
_pid_col = 'projeto_id'

if _gp_col and _tp_col and _pid_col in raw_acoes_df.columns:
    _proj_base = (
        raw_acoes_df[['atividade_id', _pid_col]]
        .dropna(subset=[_pid_col])
        .merge(raw_projetos_df[[_pid_col, _gp_col, _tp_col]], on=_pid_col, how='left')
        .dropna(subset=[_gp_col])
    )
    _rows = []
    for _, row in _proj_base.iterrows():
        grupos = [g.strip() for g in str(row[_gp_col]).split('|')] if pd.notna(row[_gp_col]) else []
        nomes  = [n.strip() for n in str(row[_tp_col]).split('|')] if pd.notna(row[_tp_col]) else []
        n_items = max(len(grupos), len(nomes))
        grupos += [''] * (n_items - len(grupos))
        nomes  += [''] * (n_items - len(nomes))
        _rows.extend(
            {'atividade_id': row['atividade_id'], 'tag_grupo': g, 'tag_nome': n}
            for g, n in zip(grupos, nomes)
        )
    _p = pd.DataFrame(_rows)
    _p = _transform_tags(_p)
    _p['origem'] = 'projeto'
else:
    _p = pd.DataFrame(columns=['atividade_id', 'areaprogTag', 'origem'])


# -- Combinar: prioridade areaprog > tag > projeto --
_priority = {'areaprog': 0, 'tag': 1, 'projeto': 2}
area_prog_tag_df = (
    pd.concat(
        [_pt1[['atividade_id', 'origem', 'areaprogTag']],
         _t[['atividade_id', 'origem', 'areaprogTag']],
         _p[['atividade_id', 'origem', 'areaprogTag']]],
        ignore_index=True
    )
    .sort_values('origem', key=lambda s: s.map(_priority))
    .drop_duplicates(subset=['atividade_id', 'areaprogTag'], keep='first')
    .reset_index(drop=True)
)

# -- Limpeza e normalização pós-concat --
for _o, _n in [('Trabalho Social com Pessoas Idosas', 'Pessoas Idosas')]:
    area_prog_tag_df['areaprogTag'] = area_prog_tag_df['areaprogTag'].str.replace(_o, _n, regex=False)
area_prog_tag_df = (
    area_prog_tag_df[
        area_prog_tag_df['areaprogTag'].notna() &
        ~area_prog_tag_df['areaprogTag'].isin({'null', 'nan', 'None'}) &
        area_prog_tag_df['areaprogTag'].str.strip().ne('')
    ]
    .drop_duplicates(subset=['atividade_id', 'areaprogTag'])
    .reset_index(drop=True)
)

print(f'area_prog_tag_df: {area_prog_tag_df.shape}')
print(f'atividades unicas: {area_prog_tag_df["atividade_id"].nunique()}')
print(f'areaprogTag unicos: {area_prog_tag_df["areaprogTag"].nunique()}')
print(area_prog_tag_df['origem'].value_counts())

## 14. Salvar em lake_gold_siplan


In [ ]:
# corrigir os nulos na hora em que aparecem: tem na tabela base mas não na de sessoes
tabela_base_df = tabela_base_df[tabela_base_df['PrimeiraData'].notna()]
print(f'tabela_base_df (após filtro sem PrimeiraData): {tabela_base_df.shape}')

save_gold(tabela_base_df,       f'base{_all}')
save_gold(raw_datas_sessoes_df, f'datas_sessoes{_all}')
save_gold(contracts_df,         f'contratos{_all}')
save_gold(solicitacoes_df,      f'solicitacoes{_all}')
save_gold(dim_gerencia_df,   'dim_gerencia')
save_gold(ponte_gerencia_df, f'ponte_gerencia{_all}')
save_gold(raw_pcap_df,          f'pcap{_all}')
acessibilidade_df = raw_acessibilidade_df[['atividade_id', 'a.identificacao']].copy()
save_gold(acessibilidade_df,    f'acessibilidade{_all}')
save_gold(raw_parcelas_df,      'parcelas')
save_gold(area_prog_tag_df,     f'area_prog_tag{_all}')
print('Salvo em lake_gold_fatos: base, datas_sessoes, contratos, solicitacoes, pcap, acessibilidade, parcelas, area_prog_tag')
